# 01. Inventario y estructura de los archivos FAERS

## Análisis temporal y geográfico de señales de farmacovigilancia en FAERS

Datos: https://fis.fda.gov/extensions/FPD-QDE-FAERS/FPD-QDE-FAERS.html

### Objetivo del notebook

El objetivo de este primer notebook es realizar una inspección sistemática de los
archivos descargados del **FDA Adverse Event Reporting System (FAERS)** antes de
comenzar su procesamiento.

FAERS es un sistema de reporte espontáneo administrado por la *U.S. Food and Drug
Administration* (FDA), que contiene reportes de eventos adversos y errores de
medicación asociados con medicamentos y productos biológicos de uso humano.

Los datos públicos se distribuyen mediante los llamados **Quarterly Data Extracts
(QDE)**. Cada extracto corresponde, en términos generales, a los reportes incluidos
en un trimestre determinado.

Para este proyecto se dispone inicialmente de los siguientes seis trimestres:

- 2025 Q1,
- 2025 Q2,
- 2025 Q3,
- 2025 Q4,
- 2026 Q1,
- 2026 Q2.

Por lo tanto, el número inicial de periodos temporales disponibles es $T = 6$.


El conjunto de trimestres analizados puede representarse como

$$
\mathcal{Q}
=
\{
2025Q1,\,
2025Q2,\,
2025Q3,\,
2025Q4,\,
2026Q1,\,
2026Q2
\}.
$$

Disponer de varios trimestres consecutivos permitirá posteriormente estudiar no
solamente qué combinaciones medicamento--evento adverso presentan señales de
desproporcionalidad, sino también **cómo dichas señales evolucionan en el tiempo**.

En particular, más adelante será posible identificar señales:

- aisladas;
- recurrentes;
- persistentes;
- emergentes.

Sin embargo, antes de realizar cualquier análisis estadístico es necesario conocer
con precisión cómo están organizados los archivos descargados.



## ¿Por qué comenzar con un inventario?

Los archivos de FAERS pueden contener una cantidad muy grande de reportes. Por esta
razón, no resulta conveniente comenzar leyendo directamente todos los archivos XML
en memoria.

Primero debemos verificar:

1. dónde se encuentra el directorio principal del proyecto;
2. qué carpetas trimestrales están disponibles;
3. si los seis trimestres esperados fueron correctamente descargados;
4. si cada trimestre contiene su correspondiente directorio `XML`;
5. posteriormente, cuántos archivos XML existen dentro de cada trimestre;
6. cuánto espacio ocupan dichos archivos;
7. si existe consistencia en los nombres y estructura de los archivos entre
   diferentes trimestres.

Esta inspección inicial permitirá diseñar posteriormente un procedimiento de lectura
que sea reproducible y eficiente en memoria.

En este notebook **todavía no se procesarán los reportes individuales**. El objetivo
es conocer y validar la estructura física de los datos antes de construir el
pipeline de extracción.

# Conceptos fundamentales para comprender este notebook

Antes de comenzar con el procesamiento de los archivos es conveniente conocer algunos
conceptos básicos de farmacovigilancia, FAERS y la estructura de los datos.

Esta sección funciona como una pequeña guía de referencia para interpretar las
variables y procedimientos utilizados a lo largo del notebook.



## 1. Farmacovigilancia

La **farmacovigilancia** es el conjunto de actividades destinadas a detectar, evaluar
y estudiar posibles problemas relacionados con el uso de medicamentos.

Uno de sus objetivos es identificar posibles asociaciones entre un medicamento y un
evento adverso que posteriormente puedan ser estudiadas con mayor profundidad.

Es importante distinguir entre:

**Evento adverso:** acontecimiento médico no deseado ocurrido durante el uso de un
medicamento.

**Reacción adversa:** evento que se reporta como posiblemente relacionado con un
medicamento.

**Señal de farmacovigilancia:** asociación medicamento--evento que presenta un patrón
de reporte suficientemente llamativo como para justificar una investigación
posterior.

Una señal estadística **no demuestra causalidad**. Indica únicamente que una
combinación merece ser estudiada.


## 2. ¿Qué es FAERS?

**FAERS** significa *FDA Adverse Event Reporting System*.

Es una base de datos de la FDA que contiene reportes espontáneos relacionados con:

- eventos adversos;
- medicamentos;
- productos biológicos;
- errores de medicación.

La unidad fundamental de FAERS es el **reporte de seguridad**.

En los archivos XML cada reporte está contenido dentro de:

`<safetyreport>`

y posee un identificador denominado:

`safetyreportid`.

Un mismo reporte puede contener:

- uno o varios medicamentos;
- una o varias reacciones adversas.

Por tanto, la estructura no es una tabla simple de una fila por paciente.

Puede representarse aproximadamente como

$$
\text{safetyreport}
\longrightarrow
\text{patient}
\longrightarrow
\begin{cases}
\text{drug}_1,\ldots,\text{drug}_{m_i},\\
\text{reaction}_1,\ldots,\text{reaction}_{n_i}.
\end{cases}
$$

donde $m_i$ es el número de registros de medicamentos y $n_i$ el número de
reacciones del reporte $i$.



## 3. Quarterly Data Extract (QDE)

Los datos públicos de FAERS se distribuyen en **extractos trimestrales**, conocidos
como *Quarterly Data Extracts* o **QDE**.

Por ejemplo,

`2025Q1`

significa:

- año: 2025;
- trimestre: 1;
- meses aproximados: enero, febrero y marzo.

En este proyecto se utilizarán inicialmente seis periodos:

$$
2025Q1,\;
2025Q2,\;
2025Q3,\;
2025Q4,\;
2026Q1,\;
2026Q2.
$$

El trimestre del archivo de procedencia se almacenará posteriormente en la variable

`qde_period`.



## 4. XML

**XML** (*Extensible Markup Language*) es un formato jerárquico para almacenar
información.

A diferencia de una tabla convencional, donde existen filas y columnas, en XML la
información se organiza mediante elementos anidados.

Por ejemplo:

    <safetyreport>
        ...
        <patient>
            <drug>...</drug>
            <drug>...</drug>
            <reaction>...</reaction>
        </patient>
    </safetyreport>

Por esta razón, un mismo `safetyreport` puede generar varias filas cuando se
construyen las tablas de medicamentos y reacciones.



## 5. Lectura incremental

Los archivos XML utilizados en este proyecto pueden ocupar cientos de megabytes.

Cargar un archivo completo en memoria puede ser ineficiente.

Por ello se utiliza

`ET.iterparse()`

para realizar una **lectura incremental**.

La idea es:

1. leer un reporte;
2. extraer las variables necesarias;
3. almacenar la información;
4. liberar el elemento XML de memoria;
5. continuar con el siguiente reporte.

De esta manera es posible procesar archivos grandes sin mantenerlos completos en
memoria.



# 6. Variables principales del reporte

| Variable | Significado |
|---|---|
| `safetyreportid` | Identificador del reporte de seguridad. Es la variable principal para relacionar reportes, medicamentos y reacciones. |
| `safetyreportversion` | Número de versión del reporte. Un caso puede recibir actualizaciones y generar versiones posteriores. |
| `primarysourcecountry` | País asociado con la fuente primaria del reporte. |
| `occurcountry` | País donde ocurrió el evento reportado. Será importante para el análisis geográfico posterior. |
| `reporttype` | Tipo de reporte registrado en FAERS. |
| `serious` | Indicador relacionado con la seriedad del reporte. |
| `receivedate` | Fecha histórica asociada con la recepción inicial del caso. |
| `receiptdate` | Fecha de recepción de la versión del reporte. En este proyecto será la principal referencia temporal. |
| `companynumb` | Identificador asignado al caso por la compañía o fuente correspondiente. |

Es especialmente importante distinguir

`receivedate`

de

`receiptdate`.

En este proyecto se encontró que `receivedate` puede corresponder a varios años
antes del trimestre actual, mientras que `receiptdate` coincide casi siempre con el
periodo del extracto.

Por esta razón se definirá

$$
\texttt{analysis\_date}
=
\texttt{receiptdate}.
$$



# 7. Variables de medicamentos

Cada elemento `<drug>` representa un registro de información sobre un medicamento.

| Variable | Significado |
|---|---|
| `drug_n` | Número consecutivo asignado al elemento `<drug>` dentro del reporte durante la extracción. |
| `drugcharacterization` | Papel que desempeña el medicamento dentro del reporte. |
| `medicinalproduct` | Nombre del producto medicinal reportado; puede ser un nombre comercial. |
| `activesubstancename` | Nombre de la sustancia o ingrediente activo. |
| `drugindication` | Indicación o motivo por el cual se utilizó el medicamento. |

En este notebook se trabaja con la siguiente interpretación de
`drugcharacterization`:

| Código | Papel |
|---:|---|
| `1` | Suspect |
| `2` | Concomitant |
| `3` | Interacting |
| `4` | Drug not administered |

**Suspect** indica que el medicamento fue reportado con un papel sospechoso respecto
al evento.

**Concomitant** indica que el paciente también utilizaba el medicamento, pero éste no
fue señalado de la misma manera como sospechoso.

Para el análisis principal se trabajará inicialmente con

$$
\texttt{drugcharacterization}=1.
$$



## 8. Producto medicinal y sustancia activa

Estas dos variables no representan necesariamente lo mismo.

Por ejemplo:

    DUPIXENT  -> DUPILUMAB
    REVLIMID  -> LENALIDOMIDE
    NOVOLOG   -> INSULIN ASPART

`medicinalproduct` puede contener el nombre del producto comercial, mientras que
`activesubstancename` identifica el principio activo.

Para reducir la fragmentación provocada por diferentes nombres comerciales se
construye:

`drug_key`

mediante

$$
\texttt{drug\_key}
=
\begin{cases}
\texttt{activesubstancename},
&
\text{si está disponible},
\\[4pt]
\texttt{medicinalproduct},
&
\text{en otro caso}.
\end{cases}
$$

Por tanto, `drug_key` será el identificador analítico del medicamento.



# 9. Variables de reacciones adversas

Cada elemento `<reaction>` representa un evento o reacción reportada.

| Variable | Significado |
|---|---|
| `reaction_n` | Número consecutivo de la reacción dentro del reporte durante la extracción. |
| `reactionmeddrapt` | Término preferido de MedDRA utilizado para identificar la reacción adversa. |
| `reactionmeddraversionpt` | Versión de MedDRA utilizada para codificar el término. |
| `reactionoutcome` | Información codificada sobre el desenlace de la reacción. |



## 10. ¿Qué es MedDRA?

**MedDRA** significa *Medical Dictionary for Regulatory Activities*.

Es un vocabulario médico estandarizado utilizado para codificar eventos y
condiciones clínicas en farmacovigilancia.

Un concepto importante es el **Preferred Term (PT)**.

Un PT es un término estandarizado utilizado para representar una reacción.

Por ejemplo, en lugar de trabajar con distintas formas libres de escribir una
reacción, FAERS utiliza términos MedDRA normalizados.

En este proyecto el PT aparece en:

`reactionmeddrapt`.

Esta variable será posteriormente el identificador principal del evento adverso:

$$
\texttt{reaction\_pt}
\equiv
\texttt{reactionmeddrapt}.
$$



# 11. Multiplicidad y deduplicación

Un punto fundamental de FAERS es que:

> una fila `<drug>` no equivale necesariamente a una exposición independiente.

Un mismo medicamento puede aparecer varias veces dentro del mismo reporte debido a
diferencias en:

- dosis;
- duración;
- indicación;
- vía de administración;
- otras características del tratamiento.

Por ello, para el análisis no se contará directamente el número de elementos
`<drug>`.

La unidad analítica del medicamento será

$$
(\texttt{safetyreportid},\texttt{drug\_key}).
$$

Es decir, un medicamento puede contribuir como máximo una vez dentro de cada
reporte.

Posteriormente, al incorporar las reacciones, la unidad fundamental será

$$
(
\texttt{safetyreportid},
\texttt{drug\_key},
\texttt{reaction\_pt}
).
$$

La **deduplicación** consiste en garantizar que cada una de estas combinaciones
aparezca una sola vez en la tabla utilizada para el análisis estadístico.



# 12. Variables derivadas utilizadas en el notebook

Además de las variables originales de FAERS, el notebook construye variables
auxiliares.

### Variables de multiplicidad

| Variable | Significado |
|---|---|
| `n_drug` | Número de elementos `<drug>` del reporte. |
| `n_active_unique` | Número de sustancias activas diferentes. |
| `n_drug_repetidos` | Diferencia entre registros de medicamentos y sustancias activas únicas. |
| `n_reaction` | Número de elementos `<reaction>`. |
| `n_reaction_unique` | Número de términos MedDRA diferentes. |
| `n_xml_rows` | Número de filas XML correspondientes al mismo par reporte--medicamento. |

Por ejemplo,

$$
\texttt{n\_drug\_repetidos}
=
\texttt{n\_drug}
-
\texttt{n\_active\_unique}.
$$



### Variables analíticas del medicamento

| Variable | Significado |
|---|---|
| `drug_key_raw` | Primera versión del identificador analítico del medicamento. |
| `drug_key_source` | Indica si el identificador provino de `activesubstancename` o del respaldo `medicinalproduct`. |
| `drug_key` | Identificador definitivo utilizado para representar el medicamento. |



### Variables temporales

| Variable | Significado |
|---|---|
| `receivedate_dt` | `receivedate` convertido a fecha de Python. |
| `receiptdate_dt` | `receiptdate` convertido a fecha de Python. |
| `receivedate_quarter` | Trimestre calendario correspondiente a `receivedate`. |
| `receiptdate_quarter` | Trimestre calendario correspondiente a `receiptdate`. |
| `qde_period` | Trimestre del archivo QDE de donde proviene el reporte. |
| `analysis_date` | Fecha principal del análisis; se define como `receiptdate`. |
| `analysis_quarter` | Trimestre calculado a partir de `analysis_date`. |
| `delta_dias` | Diferencia en días entre `receiptdate` y `receivedate`. |
| `case_history_days` | Antigüedad temporal del caso medida entre ambas fechas. |
| `temporal_match` | Indica si `analysis_quarter` coincide con `qde_period`. |
| `has_followup_history` | Indica si `receiptdate` es posterior a `receivedate`. |
| `in_study_window` | Indica si el reporte pertenece al periodo temporal definido para el estudio. |

La referencia temporal principal queda definida como

$$
\boxed{
\texttt{analysis\_date}
=
\texttt{receiptdate}
}
$$

y

$$
\boxed{
\texttt{analysis\_quarter}
=
\operatorname{Quarter}(\texttt{receiptdate})
}.
$$



# 13. Caso, reporte y versión

Una idea especialmente importante para este proyecto es que un reporte puede ser
actualizado.

Por ello debemos distinguir:

- **caso:** historia asociada con un reporte;
- **`safetyreportid`:** identificador del caso/reporte;
- **`safetyreportversion`:** versión de la información disponible.

Un valor elevado de `safetyreportversion` indica que el caso ha recibido múltiples
actualizaciones.

Esta característica será importante cuando se unan diferentes trimestres, porque
deberá comprobarse si un mismo `safetyreportid` reaparece posteriormente con una
versión nueva.



# 14. Idea general que debe conservarse

El flujo conceptual de los datos puede resumirse como

$$
\boxed{
\text{archivo QDE}
\rightarrow
\text{safetyreport}
\rightarrow
\begin{cases}
\text{medicamentos},\\
\text{reacciones}
\end{cases}
\rightarrow
\text{limpieza}
\rightarrow
\text{deduplicación}
\rightarrow
\text{pares medicamento--evento}
}
$$

Este primer notebook se concentra principalmente en las primeras etapas:

**comprender, validar y organizar los datos antes de realizar el análisis
farmacovigilante.**

La idea central es que antes de calcular cualquier señal estadística debemos estar
seguros de qué representa cada registro, qué variables identifican al medicamento y
al evento, cómo se manejan las repeticiones y qué fecha representa correctamente el
periodo de análisis.

## 1. Verificación del directorio de trabajo y detección de los trimestres

El primer paso consiste en comprobar que Python está trabajando dentro del
directorio correcto.

La estructura esperada del proyecto es aproximadamente:

    Temporal_FAERS/
    │
    ├── faers_xml_2025q1/
    ├── faers_xml_2025q2/
    ├── faers_xml_2025q3/
    ├── faers_xml_2025q4/
    ├── faers_xml_2026q1/
    ├── faers_xml_2026q2/
    │
    └── 01_inventario_y_estructura_FAERS.ipynb

Cada carpeta trimestral contiene, a su vez, archivos de documentación y un
directorio denominado `XML`.

Como el notebook se encuentra dentro de la carpeta principal `Temporal_FAERS`,
podemos obtener automáticamente esta ubicación mediante `Path.cwd()`.

Después buscaremos directorios cuyos nombres satisfagan el patrón

$$
\texttt{faers\_xml\_YYYYqQ},
$$

donde:

- $YYYY$ representa el año;
- $Q \in \{1,2,3,4\}$ representa el trimestre.

Para este proyecto esperamos encontrar exactamente $T=6$ directorios trimestrales.

La detección automática es preferible a escribir manualmente las seis rutas porque
hará que el código sea más reproducible. Si posteriormente se incorporan nuevos
trimestres, será sencillo extender el análisis sin modificar toda la estructura del
programa.

En esta primera inspección también verificaremos si cada carpeta trimestral contiene
un subdirectorio llamado `XML`.

### Salida esperada

El código mostrará:

1. la ruta desde donde se está ejecutando el notebook;
2. el número de carpetas trimestrales detectadas;
3. el año y trimestre asociados con cada carpeta;
4. si existe el directorio `XML`;
5. si falta alguno de los seis trimestres esperados.

Si la estructura es correcta, deberíamos obtener seis filas y la columna
`XML_existe` debería tomar el valor `True` en todos los casos.

In [4]:
from pathlib import Path
import re
import pandas as pd

# 1. Directorio principal del proyecto
BASE_DIR = Path.cwd()

print("Directorio de trabajo:")
print(BASE_DIR)
print()


# 2. Patrón utilizado para reconocer carpetas trimestrales
patron_trimestre = re.compile(
    r"faers_xml_(\d{4})q([1-4])",
    flags=re.IGNORECASE
)


# 3. Trimestres que esperamos encontrar
trimestres_esperados = [
    "faers_xml_2025q1",
    "faers_xml_2025q2",
    "faers_xml_2025q3",
    "faers_xml_2025q4",
    "faers_xml_2026q1",
    "faers_xml_2026q2",
]


# 4. Detectar automáticamente las carpetas trimestrales
registros = []

for carpeta in BASE_DIR.iterdir():

    if not carpeta.is_dir():
        continue

    coincidencia = patron_trimestre.fullmatch(carpeta.name)

    if coincidencia is None:
        continue

    anio = int(coincidencia.group(1))
    trimestre = int(coincidencia.group(2))

    xml_dir = carpeta / "XML"

    registros.append(
        {
            "carpeta": carpeta.name,
            "anio": anio,
            "trimestre": trimestre,
            "periodo": f"{anio}Q{trimestre}",
            "XML_existe": xml_dir.is_dir(),
        }
    )


# 5. Construir tabla resumen
df_trimestres = pd.DataFrame(registros)

if len(df_trimestres) > 0:

    df_trimestres = (
        df_trimestres
        .sort_values(["anio", "trimestre"])
        .reset_index(drop=True)
    )


# 6. Mostrar resultados
print(f"Carpetas trimestrales detectadas: {len(df_trimestres)}")
print()

display(df_trimestres)


# 7. Comprobar si falta algún trimestre esperado
carpetas_detectadas = set(df_trimestres["carpeta"].str.lower())

faltantes = [
    carpeta
    for carpeta in trimestres_esperados
    if carpeta.lower() not in carpetas_detectadas
]

print()
print("Verificación de los seis trimestres esperados:")

if len(faltantes) == 0:
    print("Se encontraron todos los trimestres esperados.")
else:
    print("Faltan las siguientes carpetas:")
    for carpeta in faltantes:
        print("  -", carpeta)


# 8. Comprobar los directorios XML
if len(df_trimestres) > 0:

    n_xml = int(df_trimestres["XML_existe"].sum())

    print()
    print(
        f"Directorios XML encontrados: "
        f"{n_xml} de {len(df_trimestres)}"
    )

Directorio de trabajo:
/Users/jamc/Desktop/Temporal_FAERS

Carpetas trimestrales detectadas: 6



,carpeta,anio,trimestre,periodo,XML_existe
0,faers_xml_2025q1,2025,1,2025Q1,True
1,faers_xml_2025q2,2025,2,2025Q2,True
2,faers_xml_2025q3,2025,3,2025Q3,True
3,faers_xml_2025q4,2025,4,2025Q4,True
4,faers_xml_2026q1,2026,1,2026Q1,True
5,faers_xml_2026q2,2026,2,2026Q2,True



Verificación de los seis trimestres esperados:
Se encontraron todos los trimestres esperados.

Directorios XML encontrados: 6 de 6


## 2. Inventario de los archivos contenidos en los directorios XML

Una vez comprobado que los seis trimestres se encuentran disponibles y que cada
uno contiene su correspondiente directorio `XML`, el siguiente paso consiste en
examinar los archivos almacenados dentro de dichos directorios.

En esta etapa todavía **no se leerá el contenido interno de los archivos XML**.
Únicamente construiremos un inventario de archivos.

Esta separación es importante porque los archivos FAERS pueden ser de gran tamaño.
Antes de decidir cómo procesarlos debemos conocer:

- cuántos archivos existen por trimestre;
- cómo se llaman;
- qué extensiones utilizan;
- cuánto ocupa cada archivo;
- cuánto ocupa en total cada trimestre;
- si todos los trimestres presentan una estructura semejante.

La documentación de los *Quarterly Data Extracts* de FAERS indica que los datos XML
se distribuyen dentro del directorio correspondiente al formato XML. Sin embargo,
la estructura física exacta de los archivos puede cambiar entre versiones o
periodos, por lo que conviene verificar directamente los archivos descargados.

### Tamaño de los archivos

El tamaño de cada archivo se expresará inicialmente en megabytes mediante

$$
\text{MB}
=
\frac{\text{bytes}}{1024^2}.
$$

También calcularemos el tamaño total de cada trimestre en gigabytes:

$$
\text{GB}
=
\frac{\text{bytes}}{1024^3}.
$$

Esta información será particularmente importante para seleccionar posteriormente
la estrategia de lectura.

Si los archivos XML tienen tamaños grandes, no será recomendable cargarlos
completamente en memoria. En ese caso utilizaremos un procedimiento de lectura
incremental, procesando los reportes uno por uno o por bloques.

### Objetivo de esta etapa

Al finalizar esta sección tendremos dos tablas:

1. un inventario detallado con un registro por archivo;
2. un resumen por trimestre con el número de archivos y el espacio ocupado.

Estas tablas permitirán conocer la dimensión física inicial del conjunto de datos
antes de analizar su estructura XML.

In [7]:
# 2. Inventario de archivos contenidos en los directorios XML

registros_archivos = []

for _, fila in df_trimestres.iterrows():

    carpeta_trimestre = BASE_DIR / fila["carpeta"]
    xml_dir = carpeta_trimestre / "XML"

    # Recorremos todos los archivos contenidos dentro de XML, incluyendo posibles subdirectorios.
    for archivo in sorted(xml_dir.rglob("*")):

        if archivo.is_file():

            size_bytes = archivo.stat().st_size

            registros_archivos.append(
                {
                    "periodo": fila["periodo"],
                    "anio": fila["anio"],
                    "trimestre": fila["trimestre"],
                    "archivo": archivo.name,
                    "extension": archivo.suffix.lower(),
                    "size_bytes": size_bytes,
                    "size_mb": size_bytes / (1024**2),
                    "ruta": str(archivo),
                }
            )


# Construir DataFrame con el inventario

df_archivos = pd.DataFrame(registros_archivos)

df_archivos = (
    df_archivos
    .sort_values(["anio", "trimestre", "archivo"])
    .reset_index(drop=True)
)


# Mostrar inventario detallado

print(f"Número total de archivos encontrados: {len(df_archivos):,}")
print()

display(
    df_archivos[
        [
            "periodo",
            "archivo",
            "extension",
            "size_mb"
        ]
    ]
)


# Resumen por trimestre
resumen_archivos = (
    df_archivos
    .groupby(
        ["anio", "trimestre", "periodo"],
        as_index=False
    )
    .agg(
        n_archivos=("archivo", "count"),
        total_bytes=("size_bytes", "sum"),
        archivo_mayor_mb=("size_mb", "max")
    )
)


# Convertir tamaño total a GB
resumen_archivos["total_gb"] = (
    resumen_archivos["total_bytes"] / (1024**3)
)


# Ordenar columnas
resumen_archivos = resumen_archivos[
    [
        "periodo",
        "n_archivos",
        "total_gb",
        "archivo_mayor_mb"
    ]
]


print("\nResumen por trimestre:")
display(resumen_archivos)


# Tamaño total de los seis trimestres
total_bytes = df_archivos["size_bytes"].sum()
total_gb = total_bytes / (1024**3)

print()
print(f"Tamaño total de los directorios XML: {total_gb:.2f} GB")

Número total de archivos encontrados: 30



,periodo,archivo,extension,size_mb
0,2025Q1,1_ADR25Q1.xml,.xml,640.353873
1,2025Q1,2_ADR25Q1.xml,.xml,703.806434
2,2025Q1,3_ADR25Q1.xml,.xml,851.004485
3,2025Q1,XML25Q1.pdf,.pdf,0.099716
4,2025Q1,XML_NTS.pdf,.pdf,0.206039
5,2025Q2,1_ADR25Q2.xml,.xml,614.683608
6,2025Q2,2_ADR25Q2.xml,.xml,659.955387
7,2025Q2,3_ADR25Q2.xml,.xml,803.076663
8,2025Q2,XML25Q2.pdf,.pdf,0.071903
9,2025Q2,XML_NTS.pdf,.pdf,0.206039



Resumen por trimestre:


,periodo,n_archivos,total_gb,archivo_mayor_mb
0,2025Q1,5,2.144014,851.004485
1,2025Q2,5,2.029291,803.076663
2,2025Q3,5,2.317160,816.159218
3,2025Q4,5,2.086153,799.763154
4,2026Q1,5,2.065181,747.854733
5,2026Q2,5,2.114662,815.977750



Tamaño total de los directorios XML: 12.76 GB


## 3. Identificación de los archivos XML de datos y verificación de la fragmentación trimestral

El inventario anterior mostró que cada directorio trimestral contiene cinco archivos:

- tres archivos con extensión `.xml`;
- un archivo PDF específico del trimestre;
- un archivo `XML_NTS.pdf` con documentación técnica.

Para el procesamiento de los reportes de farmacovigilancia únicamente nos interesan,
por ahora, los archivos con extensión `.xml`.

En los seis trimestres disponibles se observó que los datos no están contenidos en
un único archivo XML, sino que cada trimestre se encuentra dividido físicamente en
tres archivos.

Por ejemplo, para el primer trimestre de 2025 se tienen:

    1_ADR25Q1.xml
    2_ADR25Q1.xml
    3_ADR25Q1.xml

Estos tres archivos deben interpretarse como partes del mismo extracto trimestral.

Por lo tanto, si cada uno de los $T=6$ trimestres contiene $m=3$ archivos XML,
el número esperado de archivos de datos es

$$
N_{\mathrm{XML}} = T \times m = 6 \times 3 = 18.
$$

Antes de comenzar a leer el contenido de los XML verificaremos formalmente:

1. que existen exactamente $18$ archivos XML;
2. que cada trimestre contiene exactamente $3$ archivos;
3. que las partes están numeradas como $1$, $2$ y $3$;
4. que no existen archivos XML inesperados;
5. el tamaño total de datos XML correspondiente a cada trimestre.

Esta comprobación es importante porque posteriormente los tres fragmentos de un
mismo trimestre deberán procesarse conjuntamente. El trimestre será la unidad
temporal del análisis, no cada archivo individual.

Además, debido a que los archivos individuales tienen tamaños del orden de cientos
de megabytes, su lectura se realizará posteriormente de manera incremental. Esto
permitirá procesar un reporte a la vez sin mantener todo el contenido XML
simultáneamente en memoria.

In [10]:
# 3. Selección y verificación de los archivos XML de datos

# Nos quedamos únicamente con archivos cuya extensión sea .xml
df_xml = (
    df_archivos[
        df_archivos["extension"] == ".xml"
    ]
    .copy()
    .reset_index(drop=True)
)


# Extraer el número de fragmento a partir del nombre
#
# Ejemplo:
# 1_ADR25Q1.xml  -> parte = 1
# 2_ADR25Q1.xml  -> parte = 2

df_xml["parte"] = (
    df_xml["archivo"]
    .str.extract(r"^(\d+)_", expand=False)
    .astype("Int64")
)


# Ordenar los archivos cronológicamente y por número de parte
df_xml = (
    df_xml
    .sort_values(
        ["anio", "trimestre", "parte"]
    )
    .reset_index(drop=True)
)


# Mostrar inventario de XML

print(f"Número total de archivos XML: {len(df_xml)}")
print()

display(
    df_xml[
        [
            "periodo",
            "parte",
            "archivo",
            "size_mb"
        ]
    ]
)


# Resumen de fragmentos por trimestre

resumen_xml = (
    df_xml
    .groupby("periodo", as_index=False)
    .agg(
        n_xml=("archivo", "count"),
        partes=("parte", lambda x: sorted(x.dropna().tolist())),
        total_mb=("size_mb", "sum")
    )
)


resumen_xml["total_gb"] = resumen_xml["total_mb"] / 1024


print("\nResumen de archivos XML por trimestre:")
display(
    resumen_xml[
        [
            "periodo",
            "n_xml",
            "partes",
            "total_gb"
        ]
    ]
)


# Verificaciones automáticas

numero_esperado_xml = 18
partes_esperadas = [1, 2, 3]

print("\nVerificaciones:")

# 1. Número total de XML
if len(df_xml) == numero_esperado_xml:
    print(
        f"Número total correcto de XML: "
        f"{len(df_xml)} de {numero_esperado_xml}."
    )
else:
    print(
        f"Se esperaban {numero_esperado_xml} XML, "
        f"pero se encontraron {len(df_xml)}."
    )


# 2. Tres XML por trimestre
conteos_correctos = (resumen_xml["n_xml"] == 3).all()

if conteos_correctos:
    print("Todos los trimestres contienen exactamente 3 archivos XML.")
else:
    print("Algún trimestre no contiene exactamente 3 archivos XML.")


# 3. Partes 1, 2 y 3
partes_correctas = resumen_xml["partes"].apply(
    lambda x: x == partes_esperadas
).all()

if partes_correctas:
    print("Todos los trimestres contienen las partes [1, 2, 3].")
else:
    print("La numeración de los fragmentos no es consistente.")


# Tamaño total exclusivamente de los archivos XML
total_xml_gb = df_xml["size_bytes"].sum() / (1024**3)

print()
print(f"Tamaño total exclusivamente de datos XML: "
    f"{total_xml_gb:.2f} GB")

Número total de archivos XML: 18



,periodo,parte,archivo,size_mb
0,2025Q1,1,1_ADR25Q1.xml,640.353873
1,2025Q1,2,2_ADR25Q1.xml,703.806434
2,2025Q1,3,3_ADR25Q1.xml,851.004485
3,2025Q2,1,1_ADR25Q2.xml,614.683608
4,2025Q2,2,2_ADR25Q2.xml,659.955387
5,2025Q2,3,3_ADR25Q2.xml,803.076663
6,2025Q3,1,1_ADR25Q3.xml,772.971218
7,2025Q3,2,2_ADR25Q3.xml,816.159218
8,2025Q3,3,3_ADR25Q3.xml,783.357991
9,2025Q4,1,1_ADR25Q4.xml,658.930564



Resumen de archivos XML por trimestre:


,periodo,n_xml,partes,total_gb
0,2025Q1,3,"[1, 2, 3]",2.143716
1,2025Q2,3,"[1, 2, 3]",2.029019
2,2025Q3,3,"[1, 2, 3]",2.316883
3,2025Q4,3,"[1, 2, 3]",2.085871
4,2026Q1,3,"[1, 2, 3]",2.064890
5,2026Q2,3,"[1, 2, 3]",2.114381



Verificaciones:
Número total correcto de XML: 18 de 18.
Todos los trimestres contienen exactamente 3 archivos XML.
Todos los trimestres contienen las partes [1, 2, 3].

Tamaño total exclusivamente de datos XML: 12.75 GB


Con esto queda validada la estructura física de los datos:

- $T=6$ trimestres consecutivos.
- $3$ fragmentos XML por trimestre.
- $18$ archivos XML en total.
- $12.75$ GB de datos XML.
- No faltan fragmentos y la numeración es consistente.

## 4. Inspección de la estructura interna de un reporte FAERS

Hasta este punto solamente se ha estudiado la organización física de los archivos.
El siguiente paso consiste en examinar por primera vez la estructura interna de un
archivo XML.

Los archivos XML de FAERS contienen reportes individuales de seguridad, conocidos
como *Individual Case Safety Reports* (ICSR). Cada reporte está organizado de forma
jerárquica mediante etiquetas XML.

De manera esquemática, un documento XML puede imaginarse como un árbol:

$$
\text{documento}
\longrightarrow
\text{reporte}
\longrightarrow
\begin{cases}
\text{información administrativa},\\
\text{paciente},\\
\text{reacciones},\\
\text{medicamentos},\\
\text{otras características del caso}.
\end{cases}
$$

Nuestro objetivo en esta etapa **no es todavía construir una base de datos**.

Primero necesitamos responder preguntas más básicas:

1. ¿Cuál es la etiqueta raíz del archivo?
2. ¿Cómo se identifica cada reporte individual?
3. ¿Qué etiquetas aparecen directamente dentro de un reporte?
4. ¿Cómo están organizadas las reacciones?
5. ¿Cómo están organizados los medicamentos?
6. ¿Puede un reporte contener más de un medicamento?
7. ¿Puede un reporte contener más de una reacción?

Estas preguntas son fundamentales porque el conjunto de datos que construiremos
posteriormente tendrá una estructura diferente de la estructura jerárquica del XML.

Por ejemplo, conceptualmente un reporte puede contener

$$
D_i = \{d_{i1},d_{i2},\ldots,d_{im_i}\}
$$

medicamentos y

$$
A_i = \{a_{i1},a_{i2},\ldots,a_{in_i}\}
$$

reacciones adversas.

Por tanto, un único reporte puede contribuir potencialmente con varios pares

$$
(d_{ij},a_{ik}).
$$

Antes de realizar esta expansión es indispensable comprender correctamente la
jerarquía del XML.

### Estrategia de lectura

Los archivos individuales tienen tamaños aproximados entre $600$ y $850$ MB.
Por ello, no utilizaremos una instrucción que cargue todo el documento XML en
memoria.

En su lugar utilizaremos **lectura incremental** o *streaming parsing*.

La idea consiste en recorrer el documento secuencialmente hasta encontrar el
primer elemento correspondiente a un reporte:

$$
\texttt{<safetyreport>}.
$$

Una vez encontrado el primer reporte:

1. detendremos la lectura;
2. inspeccionaremos únicamente su estructura;
3. mostraremos las etiquetas encontradas;
4. liberaremos de memoria el elemento procesado.

De esta manera podremos conocer la estructura real de FAERS utilizando solamente
una fracción muy pequeña del archivo.

### Importante

En esta sección solamente utilizaremos el primer reporte de

`1_ADR25Q1.xml`.

Este reporte no se utilizará todavía para obtener resultados científicos. Se
empleará exclusivamente como ejemplo para diseñar correctamente el procedimiento
de extracción que utilizaremos después con los $18$ archivos XML.

In [12]:
import xml.etree.ElementTree as ET
from collections import Counter


# 4. Seleccionar el primer archivo XML del proyecto

primer_xml = Path(
    df_xml.loc[
        (df_xml["periodo"] == "2025Q1") &
        (df_xml["parte"] == 1),
        "ruta"
    ].iloc[0]
)

print("Archivo seleccionado:")
print(primer_xml)
print()


# Función auxiliar

def limpiar_tag(tag):
    """
    Elimina un posible namespace de una etiqueta XML.

    Ejemplo:
        {namespace}safetyreport -> safetyreport
    """
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag


# Leer incrementalmente hasta encontrar el primer safetyreport

primer_reporte = None
root_tag = None

context = ET.iterparse(
    primer_xml,
    events=("start", "end")
)

for event, elem in context:

    tag = limpiar_tag(elem.tag)

    # La primera etiqueta encontrada corresponde a la raíz
    if root_tag is None and event == "start":
        root_tag = tag

    # Detenernos cuando termina el primer safetyreport completo
    if event == "end" and tag.lower() == "safetyreport":
        primer_reporte = elem
        break


# Verificación
print(f"Etiqueta raíz del documento: {root_tag}")

if primer_reporte is None:
    print("No se encontró ningún elemento <safetyreport>.")
else:
    print("Se encontró correctamente el primer <safetyreport>.")


# Etiquetas hijas directas del primer reporte

if primer_reporte is not None:

    hijos_directos = [
        limpiar_tag(hijo.tag)
        for hijo in primer_reporte
    ]

    print("\nEtiquetas directamente contenidas en <safetyreport>:")
    
    for tag in hijos_directos:
        print(f"  - {tag}")


# Todas las etiquetas presentes dentro del primer reporte

if primer_reporte is not None:

    todas_las_etiquetas = [
        limpiar_tag(elem.tag)
        for elem in primer_reporte.iter()
    ]

    conteo_etiquetas = Counter(todas_las_etiquetas)

    df_etiquetas = pd.DataFrame(
        sorted(
            conteo_etiquetas.items(),
            key=lambda x: x[0].lower()
        ),
        columns=["etiqueta", "n_apariciones"]
    )

    print(
        "\nNúmero de etiquetas diferentes dentro "
        f"del primer reporte: {len(df_etiquetas)}"
    )

    display(df_etiquetas)

Archivo seleccionado:
/Users/jamc/Desktop/Temporal_FAERS/faers_xml_2025q1/XML/1_ADR25Q1.xml

Etiqueta raíz del documento: ichicsr
Se encontró correctamente el primer <safetyreport>.

Etiquetas directamente contenidas en <safetyreport>:
  - safetyreportversion
  - safetyreportid
  - primarysourcecountry
  - occurcountry
  - transmissiondateformat
  - transmissiondate
  - reporttype
  - serious
  - seriousnessdeath
  - seriousnesslifethreatening
  - seriousnesshospitalization
  - seriousnessdisabling
  - seriousnesscongenitalanomali
  - seriousnessother
  - receivedateformat
  - receivedate
  - receiptdateformat
  - receiptdate
  - fulfillexpeditecriteria
  - companynumb
  - primarysource
  - sender
  - receiver
  - patient

Número de etiquetas diferentes dentro del primer reporte: 59


,etiqueta,n_apariciones
0,actiondrug,18
1,activesubstance,18
2,activesubstancename,18
3,companynumb,1
4,drug,18
5,drugadditional,18
6,drugadministrationroute,9
7,drugauthorizationnumb,6
8,drugcharacterization,18
9,drugdosageform,3


La salida es muy informativa. Ya confirmamos que la estructura real del XML coincide con lo que necesitábamos para el proyecto:

* `safetyreport` es la unidad básica de reporte.
* `safetyreportid`, `occurcountry`, `receivedate` y `receiptdate` están directamente dentro del reporte.
* `patient` aparece una vez.
* Dentro de `patient` encontramos estructuras repetidas de `reaction` y `drug`.
* En **este primer reporte concreto** aparecen $18$ elementos `drug` y $3$ elementos `reaction`.

Esto último es especialmente importante ya que FAERS no tiene una estructura tabular simple. Un solo reporte puede contener múltiples medicamentos y múltiples eventos adversos. La documentación de la FDA confirma precisamente que puede haber uno o más medicamentos y uno o más términos MedDRA por reporte. 

En principio, si un reporte contiene $m_i$ medicamentos y $n_i$ reacciones, podrían construirse hasta

$$
m_i n_i
$$

combinaciones medicamento–evento.

## 5. Exploración del contenido del primer reporte

La inspección de etiquetas mostró que un elemento `safetyreport` contiene información
administrativa del caso y un elemento `patient`. Dentro de este último pueden aparecer
múltiples medicamentos y múltiples reacciones adversas.

En el primer reporte analizado se encontraron:

$$
m_1 = 18
$$

registros de medicamentos y

$$
n_1 = 3
$$

registros de reacciones.

Esta característica muestra por qué los datos FAERS no pueden analizarse directamente
como una tabla convencional: existe una relación de tipo **uno a muchos** entre un
reporte y sus medicamentos, y también entre un reporte y sus reacciones.

De manera conceptual, la estructura observada puede representarse como

$$
\text{safetyreport}
\longrightarrow
\text{patient}
\longrightarrow
\begin{cases}
\text{drug}_1,\ldots,\text{drug}_{m_i},\\
\text{reaction}_1,\ldots,\text{reaction}_{n_i}.
\end{cases}
$$

Si se combinaran directamente todos los medicamentos con todas las reacciones del
reporte, el número máximo de pares medicamento--evento generados por el reporte $i$
sería

$$
N_i = m_i n_i.
$$

Sin embargo, todavía no realizaremos esa expansión.

Primero necesitamos conocer el contenido de las variables más importantes del caso y,
en particular, estudiar el papel reportado para cada medicamento.

### Información que se extraerá

Para el primer reporte se mostrarán tres bloques de información.

#### 1. Información general del reporte

Se inspeccionarán:

- `safetyreportversion`;
- `safetyreportid`;
- `primarysourcecountry`;
- `occurcountry`;
- `reporttype`;
- `serious`;
- `receivedate`;
- `receiptdate`;
- `companynumb`.

Es importante conservar inicialmente tanto `receivedate` como `receiptdate` para
estudiar posteriormente sus diferencias y decidir cuál debe utilizarse para construir
la dimensión temporal del proyecto.

#### 2. Reacciones adversas

Para cada elemento `reaction` se extraerán:

- `reactionmeddrapt`;
- `reactionmeddraversionpt`;
- `reactionoutcome`.

El campo `reactionmeddrapt` contiene el término preferido de MedDRA asociado con la
reacción reportada.

#### 3. Medicamentos

Para cada elemento `drug` se extraerán:

- `drugcharacterization`;
- `medicinalproduct`;
- `activesubstancename`;
- `drugindication`.

La variable `drugcharacterization` será particularmente importante porque describe
el papel del medicamento dentro del reporte.

Por el momento **no asignaremos significado clínico a sus códigos numéricos**.
Primero observaremos qué valores aparecen en los datos y posteriormente verificaremos
su definición utilizando la documentación técnica del formato XML.

### Objetivo de esta etapa

El objetivo es transformar un único reporte XML en pequeñas tablas legibles para
entender cómo deberán construirse posteriormente las tablas maestras de:

$$
\text{reportes},
\qquad
\text{medicamentos},
\qquad
\text{reacciones}.
$$

Todavía no se procesarán los demás reportes ni se generarán pares
medicamento--evento.

In [16]:
# 5. Exploración del contenido del primer safetyreport


# Función auxiliar para obtener el texto de una etiqueta hija
def obtener_texto(elemento, etiqueta):
    """
    Busca una etiqueta dentro de un elemento XML y devuelve
    su contenido de texto.

    Si la etiqueta no existe o está vacía, devuelve None.
    """

    for hijo in elemento.iter():

        if limpiar_tag(hijo.tag).lower() == etiqueta.lower():

            if hijo.text is None:
                return None

            texto = hijo.text.strip()

            return texto if texto != "" else None

    return None


# 5.1 Información general del reporte
campos_reporte = [
    "safetyreportversion",
    "safetyreportid",
    "primarysourcecountry",
    "occurcountry",
    "reporttype",
    "serious",
    "receivedate",
    "receiptdate",
    "companynumb",
]

datos_reporte = {
    campo: obtener_texto(primer_reporte, campo)
    for campo in campos_reporte
}

df_reporte_ejemplo = pd.DataFrame(
    datos_reporte.items(),
    columns=["variable", "valor"]
)

print("INFORMACIÓN GENERAL DEL PRIMER REPORTE")
display(df_reporte_ejemplo)


# 5.2 Localizar el elemento patient
patient = None

for elem in primer_reporte:

    if limpiar_tag(elem.tag).lower() == "patient":
        patient = elem
        break

if patient is None:
    raise ValueError(
        "No se encontró el elemento <patient> en el primer reporte."
    )


# 5.3 Extraer las reacciones

registros_reacciones = []

for elem in patient:

    if limpiar_tag(elem.tag).lower() == "reaction":

        registros_reacciones.append(
            {
                "reactionmeddrapt":
                    obtener_texto(elem, "reactionmeddrapt"),

                "reactionmeddraversionpt":
                    obtener_texto(elem, "reactionmeddraversionpt"),

                "reactionoutcome":
                    obtener_texto(elem, "reactionoutcome"),
            }
        )

df_reacciones_ejemplo = pd.DataFrame(registros_reacciones)

print()
print(
    f"REACCIONES EN EL PRIMER REPORTE: "
    f"{len(df_reacciones_ejemplo)}"
)

display(df_reacciones_ejemplo)


# 5.4 Extraer los medicamentos

registros_medicamentos = []

for elem in patient:

    if limpiar_tag(elem.tag).lower() == "drug":

        registros_medicamentos.append(
            {
                "drugcharacterization":
                    obtener_texto(elem, "drugcharacterization"),

                "medicinalproduct":
                    obtener_texto(elem, "medicinalproduct"),

                "activesubstancename":
                    obtener_texto(elem, "activesubstancename"),

                "drugindication":
                    obtener_texto(elem, "drugindication"),
            }
        )

df_medicamentos_ejemplo = pd.DataFrame(registros_medicamentos)

# Agregamos un identificador local únicamente para facilitar la lectura de la tabla.
df_medicamentos_ejemplo.insert(
    0,
    "drug_n",
    range(1, len(df_medicamentos_ejemplo) + 1)
)

print()
print(
    f"MEDICAMENTOS EN EL PRIMER REPORTE: "
    f"{len(df_medicamentos_ejemplo)}"
)

display(df_medicamentos_ejemplo)


# 5.5 Resumen de la multiplicidad del reporte
m = len(df_medicamentos_ejemplo)
n = len(df_reacciones_ejemplo)

print()
print("RESUMEN DE LA ESTRUCTURA DEL REPORTE")
print(f"Número de medicamentos (m): {m}")
print(f"Número de reacciones (n): {n}")
print(
    f"Número máximo de combinaciones medicamento-reacción "
    f"(m × n): {m * n}"
)


# 5.6 Valores observados de drugcharacterization

print()
print("VALORES OBSERVADOS DE drugcharacterization")

display(
    df_medicamentos_ejemplo[
        "drugcharacterization"
    ]
    .value_counts(dropna=False)
    .rename_axis("drugcharacterization")
    .reset_index(name="n_medicamentos")
)

INFORMACIÓN GENERAL DEL PRIMER REPORTE


,variable,valor
0,safetyreportversion,1
1,safetyreportid,24717255
2,primarysourcecountry,CN
3,occurcountry,CN
4,reporttype,1
5,serious,1
6,receivedate,20241210
7,receiptdate,20241210
8,companynumb,CN-MEITHEAL-2024MPLIT00416



REACCIONES EN EL PRIMER REPORTE: 3


,reactionmeddrapt,reactionmeddraversionpt,reactionoutcome
0,Hepatic failure,27.1,1
1,Neutropenia,27.1,3
2,Thrombocytopenia,27.1,3



MEDICAMENTOS EN EL PRIMER REPORTE: 18


,drug_n,drugcharacterization,medicinalproduct,activesubstancename,drugindication
0,1,1,ETOPOSIDE,ETOPOSIDE,T-cell lymphoma
1,2,1,ETOPOSIDE,ETOPOSIDE,Haemophagocytic lymphohistiocytosis
2,3,1,IFOSFAMIDE,IFOSFAMIDE,Haemophagocytic lymphohistiocytosis
3,4,1,IFOSFAMIDE,IFOSFAMIDE,T-cell lymphoma
4,5,1,GEMCITABINE,GEMCITABINE,T-cell lymphoma
5,6,1,GEMCITABINE,GEMCITABINE,Haemophagocytic lymphohistiocytosis
6,7,1,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,T-cell lymphoma
7,8,1,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,Haemophagocytic lymphohistiocytosis
8,9,1,DEXAMETHASONE,DEXAMETHASONE,T-cell lymphoma
9,10,1,DEXAMETHASONE,DEXAMETHASONE,Haemophagocytic lymphohistiocytosis



RESUMEN DE LA ESTRUCTURA DEL REPORTE
Número de medicamentos (m): 18
Número de reacciones (n): 3
Número máximo de combinaciones medicamento-reacción (m × n): 54

VALORES OBSERVADOS DE drugcharacterization


,drugcharacterization,n_medicamentos
0,1,18


La salida revela dos cuestiones metodológicas muy importantes antes de seguir.

Primero, en este reporte los $18$ elementos `<drug>` **no representan 18 medicamentos distintos**. A simple vista aparecen **9 sustancias diferentes**, cada una repetida dos veces porque está asociada con dos indicaciones distintas (`T-cell lymphoma` y `Haemophagocytic lymphohistiocytosis`). Por tanto, si usáramos directamente los 18 registros para construir pares fármaco–evento, estaríamos introduciendo duplicación artificial.

Segundo, ya podemos interpretar correctamente `drugcharacterization`. La especificación técnica oficial de la FDA para el XML E2B define:

* `1 = Suspect`
* `2 = Concomitant`
* `3 = Interacting`
* `4 = Drug not administered` en contextos donde aplica. ([U.S. Food and Drug Administration][1])

Esto requiere una pequeña corrección conceptual respecto de la propuesta original: **`drugcharacterization = 1` significa medicamento sospechoso, pero no distingue “Primary Suspect” de “Secondary Suspect”**. Esa distinción sí existe en los archivos ASCII mediante `ROLE_COD`, donde `PS` significa *Primary Suspect* y `SS` *Secondary Suspect*.  Como nuestro proyecto está trabajando con XML, por ahora la cohorte natural será **Suspect**, no estrictamente **Primary Suspect**.


## 6. Interpretación del papel del medicamento y estudio de registros repetidos

La exploración del primer reporte mostró una característica importante de la
estructura XML de FAERS: los elementos `<drug>` no deben interpretarse
automáticamente como medicamentos distintos.

En el reporte utilizado como ejemplo se encontraron

$$
m=18
$$

elementos `<drug>`. Sin embargo, la inspección visual de los nombres mostró que
varios medicamentos aparecen repetidos.

Por ejemplo, `ETOPOSIDE` aparece dos veces, una asociado con la indicación

`T-cell lymphoma`

y otra con

`Haemophagocytic lymphohistiocytosis`.

El mismo patrón parece presentarse para otros medicamentos del reporte.

Esto significa que debemos distinguir entre:

1. **registro XML de medicamento**;
2. **medicamento único dentro del reporte**.

Estas dos cantidades no necesariamente son iguales.


### 6.1 Papel del medicamento en el reporte

La etiqueta

`drugcharacterization`

describe el papel asignado al medicamento dentro del reporte.

De acuerdo con la especificación utilizada por FAERS, los códigos principales son:

| Código | Interpretación |
|---|---|
| 1 | Suspect |
| 2 | Concomitant |
| 3 | Interacting |
| 4 | Drug not administered |

Por tanto, cuando

$$
\texttt{drugcharacterization}=1,
$$

el medicamento fue reportado como **sospechoso**.

Es importante señalar que esta variable XML no distingue directamente entre
*Primary Suspect* y *Secondary Suspect*. Esa distinción existe en otras
representaciones de FAERS, como los archivos ASCII.

En consecuencia, mientras se trabaje exclusivamente con los archivos XML, la
cohorte principal del proyecto se definirá inicialmente mediante

$$
\mathcal{D}_{S}
=
\{
d:\texttt{drugcharacterization}(d)=1
\},
$$

es decir, el conjunto de medicamentos clasificados como **Suspect**.

Esta decisión deberá documentarse explícitamente en la metodología del estudio.



### 6.2 Medicamento reportado frente a sustancia activa

Para identificar medicamentos se dispone inicialmente de dos variables:

- `medicinalproduct`;
- `activesubstancename`.

La primera contiene el nombre del producto medicinal reportado, mientras que la
segunda contiene el nombre de la sustancia activa.

Para un análisis de desproporcionalidad es deseable evitar que una misma sustancia
sea contabilizada varias veces debido a diferencias en nombres comerciales,
presentaciones o registros repetidos.

Por esta razón, más adelante construiremos una variable estandarizada

$$
\texttt{drug\_key},
$$

que servirá como identificador analítico del medicamento.

Sin embargo, antes de definirla debemos estudiar empíricamente la correspondencia
entre `medicinalproduct` y `activesubstancename`.



### 6.3 Duplicados dentro de un reporte

Para un mismo `safetyreportid`, un medicamento puede aparecer en más de un elemento
`<drug>`.

Por ello, antes de construir pares medicamento--evento será necesario eliminar
duplicaciones a un nivel apropiado.

Una posible unidad analítica será

$$
(\texttt{safetyreportid},\texttt{drug\_key}),
$$

de manera que un medicamento determinado contribuya como máximo una vez dentro de
un mismo reporte.

Posteriormente, al incorporar las reacciones, la unidad utilizada para el análisis
de desproporcionalidad será

$$
(\texttt{safetyreportid},
\texttt{drug\_key},
\texttt{reaction\_pt}).
$$

Esta deduplicación es fundamental. De lo contrario, un medicamento que aparezca
varias veces por diferencias en indicación, dosis, vía de administración u otra
información podría aumentar artificialmente el número de combinaciones
fármaco--evento.



### Objetivo de esta sección

En esta etapa utilizaremos únicamente el primer reporte para:

1. traducir `drugcharacterization` a una etiqueta interpretable;
2. calcular cuántos registros XML de medicamentos existen;
3. calcular cuántos nombres de medicamentos son realmente distintos;
4. calcular cuántas sustancias activas son distintas;
5. identificar cuáles medicamentos están repetidos;
6. examinar por qué se producen esas repeticiones.

Todavía no se eliminarán registros del conjunto de datos completo. Primero
comprenderemos el fenómeno utilizando el reporte de ejemplo.

In [19]:
# 6. Interpretación de drugcharacterization y análisis de medicamentos repetidos

# 6.1 Diccionario para interpretar drugcharacterization

mapa_drugcharacterization = {
    "1": "Suspect",
    "2": "Concomitant",
    "3": "Interacting",
    "4": "Drug not administered"
}


df_medicamentos_ejemplo["drug_role"] = (
    df_medicamentos_ejemplo["drugcharacterization"]
    .map(mapa_drugcharacterization)
    .fillna("Unknown / other")
)


print("MEDICAMENTOS CON INTERPRETACIÓN DE SU PAPEL")
display(
    df_medicamentos_ejemplo[
        [
            "drug_n",
            "drugcharacterization",
            "drug_role",
            "medicinalproduct",
            "activesubstancename",
            "drugindication"
        ]
    ]
)


# 6.2 Número de registros y medicamentos distintos

n_registros_drug = len(df_medicamentos_ejemplo)

n_medicinalproduct = (
    df_medicamentos_ejemplo["medicinalproduct"]
    .nunique(dropna=True)
)

n_activesubstance = (
    df_medicamentos_ejemplo["activesubstancename"]
    .nunique(dropna=True)
)


print("\nRESUMEN DE UNICIDAD")

print(
    f"Registros <drug> presentes en el XML: "
    f"{n_registros_drug}"
)

print(
    f"Valores únicos de medicinalproduct: "
    f"{n_medicinalproduct}"
)

print(
    f"Valores únicos de activesubstancename: "
    f"{n_activesubstance}"
)


# 6.3 Frecuencia de cada producto medicinal

frecuencia_productos = (
    df_medicamentos_ejemplo
    .groupby(
        ["medicinalproduct", "activesubstancename"],
        dropna=False
    )
    .agg(
        n_registros=("drug_n", "count"),
        n_indicaciones=("drugindication", "nunique")
    )
    .reset_index()
    .sort_values(
        ["n_registros", "medicinalproduct"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


print("\nFRECUENCIA DE LOS MEDICAMENTOS EN EL REPORTE")
display(frecuencia_productos)


# 6.4 Mostrar exclusivamente los medicamentos repetidos

medicamentos_repetidos = (
    frecuencia_productos[
        frecuencia_productos["n_registros"] > 1
    ]
    .reset_index(drop=True)
)


print(
    "\nMEDICAMENTOS QUE APARECEN MÁS DE UNA VEZ "
    "EN EL REPORTE"
)

display(medicamentos_repetidos)


# 6.5 Examinar las indicaciones asociadas con cada medicamento

indicaciones_por_medicamento = (
    df_medicamentos_ejemplo
    .groupby(
        ["medicinalproduct", "activesubstancename"],
        dropna=False
    )["drugindication"]
    .apply(
        lambda x: sorted(
            set(
                valor
                for valor in x
                if pd.notna(valor)
            )
        )
    )
    .reset_index(name="indicaciones")
)


print("\nINDICACIONES ASOCIADAS CON CADA MEDICAMENTO")
display(indicaciones_por_medicamento)


# 6.6 Correspondencia medicinalproduct-activesubstancename

correspondencia = (
    df_medicamentos_ejemplo[
        [
            "medicinalproduct",
            "activesubstancename"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print(
    "\nCORRESPONDENCIAS ÚNICAS ENTRE "
    "medicinalproduct Y activesubstancename"
)

display(correspondencia)


# 6.7 Número de medicamentos sospechosos únicos

df_suspect_ejemplo = (
    df_medicamentos_ejemplo[
        df_medicamentos_ejemplo["drugcharacterization"] == "1"
    ]
    .copy()
)


n_suspect_registros = len(df_suspect_ejemplo)

n_suspect_unicos = (
    df_suspect_ejemplo["activesubstancename"]
    .nunique(dropna=True)
)


print("\nMEDICAMENTOS CLASIFICADOS COMO SUSPECT")

print(
    f"Registros XML clasificados como Suspect: "
    f"{n_suspect_registros}"
)

print(
    f"Sustancias activas Suspect distintas: "
    f"{n_suspect_unicos}"
)

MEDICAMENTOS CON INTERPRETACIÓN DE SU PAPEL


,drug_n,drugcharacterization,drug_role,medicinalproduct,activesubstancename,drugindication
0,1,1,Suspect,ETOPOSIDE,ETOPOSIDE,T-cell lymphoma
1,2,1,Suspect,ETOPOSIDE,ETOPOSIDE,Haemophagocytic lymphohistiocytosis
2,3,1,Suspect,IFOSFAMIDE,IFOSFAMIDE,Haemophagocytic lymphohistiocytosis
3,4,1,Suspect,IFOSFAMIDE,IFOSFAMIDE,T-cell lymphoma
4,5,1,Suspect,GEMCITABINE,GEMCITABINE,T-cell lymphoma
5,6,1,Suspect,GEMCITABINE,GEMCITABINE,Haemophagocytic lymphohistiocytosis
6,7,1,Suspect,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,T-cell lymphoma
7,8,1,Suspect,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,Haemophagocytic lymphohistiocytosis
8,9,1,Suspect,DEXAMETHASONE,DEXAMETHASONE,T-cell lymphoma
9,10,1,Suspect,DEXAMETHASONE,DEXAMETHASONE,Haemophagocytic lymphohistiocytosis



RESUMEN DE UNICIDAD
Registros <drug> presentes en el XML: 18
Valores únicos de medicinalproduct: 9
Valores únicos de activesubstancename: 9

FRECUENCIA DE LOS MEDICAMENTOS EN EL REPORTE


,medicinalproduct,activesubstancename,n_registros,n_indicaciones
0,ASPARAGINASE,ASPARAGINASE,2,2
1,CISPLATIN,CISPLATIN,2,2
2,DEXAMETHASONE,DEXAMETHASONE,2,2
3,ETOPOSIDE,ETOPOSIDE,2,2
4,GEMCITABINE,GEMCITABINE,2,2
5,IFOSFAMIDE,IFOSFAMIDE,2,2
6,METHOTREXATE,METHOTREXATE,2,2
7,PEGASPARGASE,PEGASPARGASE,2,2
8,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,2,2



MEDICAMENTOS QUE APARECEN MÁS DE UNA VEZ EN EL REPORTE


...


INDICACIONES ASOCIADAS CON CADA MEDICAMENTO


,medicinalproduct,activesubstancename,indicaciones
0,ASPARAGINASE,ASPARAGINASE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
1,CISPLATIN,CISPLATIN,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
2,DEXAMETHASONE,DEXAMETHASONE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
3,ETOPOSIDE,ETOPOSIDE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
4,GEMCITABINE,GEMCITABINE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
5,IFOSFAMIDE,IFOSFAMIDE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
6,METHOTREXATE,METHOTREXATE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
7,PEGASPARGASE,PEGASPARGASE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."
8,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE,"[Haemophagocytic lymphohistiocytosis, T-cell l..."



CORRESPONDENCIAS ÚNICAS ENTRE medicinalproduct Y activesubstancename


,medicinalproduct,activesubstancename
0,ETOPOSIDE,ETOPOSIDE
1,IFOSFAMIDE,IFOSFAMIDE
2,GEMCITABINE,GEMCITABINE
3,RUXOLITINIB PHOSPHATE,RUXOLITINIB PHOSPHATE
4,DEXAMETHASONE,DEXAMETHASONE
5,PEGASPARGASE,PEGASPARGASE
6,CISPLATIN,CISPLATIN
7,METHOTREXATE,METHOTREXATE
8,ASPARAGINASE,ASPARAGINASE



MEDICAMENTOS CLASIFICADOS COMO SUSPECT
Registros XML clasificados como Suspect: 18
Sustancias activas Suspect distintas: 9


Este primer caso ya nos dejó dos hallazgos importantes que conviene conservar.

Primero, los $18$ elementos `<drug>` corresponden realmente a **9 sustancias activas distintas**. Cada una aparece dos veces porque el mismo medicamento está asociado con dos indicaciones diferentes. Esto confirma que **no podemos contar directamente los elementos `<drug>` como exposiciones independientes**. La propia FDA advierte que un mismo producto puede generar múltiples registros de medicamento dentro de un reporte por diferencias en información asociada al producto, por lo que el programa de importación debe admitir más de una fila por medicamento. 

Segundo, hay algo incluso más interesante que apareció en la tabla general:

$$
\texttt{receiptdate}=\texttt{20241210},
$$

aunque el reporte se encuentra en el extracto **2025Q1**.

Esto **no debemos tratarlo como un error**. La documentación de FAERS explica que, debido al sistema Case/Version y al procesamiento de actualizaciones, algunos casos pueden aparecer en un extracto posterior al periodo en que originalmente fueron recibidos.  Este detalle puede ser crucial para decidir posteriormente si nuestra dimensión temporal se basa en la carpeta QDE, en `receiptdate`, o en ambas cosas con funciones distintas.

Por eso, antes de diseñar el parser definitivo, tenemos que ampliar nuestra inspección de 1 reporte a una muestra de 1,000 reportes del mismo archivo. Con eso podremos comprobar si lo observado es excepcional o habitual.

## 7. Exploración de una muestra de reportes del primer archivo XML

Hasta ahora se ha estudiado detalladamente un único reporte. Esta inspección permitió
comprender la estructura jerárquica básica de FAERS y detectar algunas características
importantes, como la existencia de múltiples registros de medicamentos para una misma
sustancia activa.

Sin embargo, un único caso no es suficiente para diseñar las reglas de procesamiento
que se aplicarán posteriormente a millones de registros.

El siguiente paso consiste en ampliar la exploración a una muestra de los primeros

$$
N=1000
$$

reportes contenidos en `1_ADR25Q1.xml`.

Esta muestra todavía no se utilizará para realizar inferencia estadística ni para
detectar señales de farmacovigilancia. Su propósito es exclusivamente
**diagnóstico y metodológico**.


### 7.1 ¿Qué queremos investigar?

Para cada reporte de la muestra registraremos:

- `safetyreportid`;
- `safetyreportversion`;
- `receivedate`;
- `receiptdate`;
- `occurcountry`;
- número de elementos `<drug>`;
- número de sustancias activas distintas;
- número de elementos `<reaction>`;
- número de términos MedDRA distintos.

Esto permitirá estudiar la multiplicidad real de los reportes.

Si para el reporte $i$ definimos

$$
m_i=
\text{número de registros de medicamentos},
$$

$$
m_i^{*}=
\text{número de sustancias activas distintas},
$$

y

$$
n_i=
\text{número de reacciones},
$$

podremos comparar $m_i$ con $m_i^{*}$.

Cuando

$$
m_i > m_i^{*},
$$

existirán registros repetidos de alguna sustancia activa dentro del mismo reporte.

Podemos definir la diferencia

$$
d_i=m_i-m_i^{*},
$$

como una medida sencilla del número de registros adicionales generados por
repeticiones.


### 7.2 Papel de los medicamentos

También construiremos una tabla independiente con los medicamentos encontrados en
los $1000$ reportes.

Esto permitirá estudiar:

- la distribución de `drugcharacterization`;
- la frecuencia de valores faltantes en `medicinalproduct`;
- la frecuencia de valores faltantes en `activesubstancename`;
- la correspondencia entre producto medicinal y sustancia activa;
- la frecuencia de registros repetidos dentro de un mismo reporte.

Por el momento no eliminaremos ninguna observación.



### 7.3 Reacciones adversas

Se construirá igualmente una tabla de las reacciones presentes en la muestra.

Para cada reacción se conservarán:

- `safetyreportid`;
- `reactionmeddrapt`;
- `reactionmeddraversionpt`;
- `reactionoutcome`.

Esto permitirá comprobar si una misma reacción puede aparecer repetida dentro de un
reporte y conocer las versiones de MedDRA presentes.


### 7.4 Fechas del reporte y trimestre del extracto

El primer caso estudiado presentó

$$
\texttt{receiptdate}=20241210,
$$

aunque se encontraba en el extracto `2025Q1`.

Por esta razón analizaremos también la distribución de `receivedate` y
`receiptdate`.

Para esta muestra queremos determinar:

1. la fecha mínima;
2. la fecha máxima;
3. cuántos reportes presentan una fecha anterior al inicio de 2025;
4. si `receivedate` y `receiptdate` son siempre iguales;
5. con qué frecuencia difieren.

Este análisis será importante porque existen al menos dos nociones temporales
distintas:

$$
\text{trimestre del archivo QDE}
$$

y

$$
\text{fecha registrada dentro del reporte}.
$$

No asumiremos todavía que ambas representan exactamente el mismo concepto.



### 7.5 Estrategia computacional

La lectura seguirá realizándose de manera incremental.

El algoritmo recorrerá el XML hasta completar $1000$ elementos
`<safetyreport>` y se detendrá inmediatamente después.

Por tanto, todavía no se procesará el archivo completo de aproximadamente
$640$ MB.

Como resultado construiremos tres tablas temporales:

$$
\texttt{df\_muestra\_reportes},
$$

$$
\texttt{df\_muestra\_medicamentos},
$$

y

$$
\texttt{df\_muestra\_reacciones}.
$$

Estas tablas servirán únicamente para comprender la estructura de los datos y
definir posteriormente las reglas de limpieza y deduplicación.

In [21]:
# 7. Exploración de una muestra de 1,000 reportes

N_MUESTRA = 1000

registros_reportes = []
registros_medicamentos = []
registros_reacciones = []


# Función auxiliar: obtener hijos directos con una etiqueta

def hijos_con_tag(elemento, etiqueta):
    """
    Devuelve los hijos directos de 'elemento' cuya etiqueta,
    una vez eliminado un posible namespace, coincide con
    'etiqueta'.
    """

    return [
        hijo
        for hijo in elemento
        if limpiar_tag(hijo.tag).lower() == etiqueta.lower()
    ]


# Lectura incremental
contador_reportes = 0

context = ET.iterparse(
    primer_xml,
    events=("end",)
)

for event, elem in context:

    if limpiar_tag(elem.tag).lower() != "safetyreport":
        continue

    # Identificador del reporte
    safetyreportid = obtener_texto(
        elem,
        "safetyreportid"
    )

    safetyreportversion = obtener_texto(
        elem,
        "safetyreportversion"
    )


    # Localizar patient
    patient = None

    for hijo in elem:

        if limpiar_tag(hijo.tag).lower() == "patient":
            patient = hijo
            break


    # Listas de medicamentos y reacciones del reporte

    drugs = []
    reactions = []

    if patient is not None:

        drugs = hijos_con_tag(
            patient,
            "drug"
        )

        reactions = hijos_con_tag(
            patient,
            "reaction"
        )


    # MEDICAMENTOS
    sustancias_activas = []

    for drug_n, drug in enumerate(drugs, start=1):

        medicinalproduct = obtener_texto(
            drug,
            "medicinalproduct"
        )

        activesubstancename = obtener_texto(
            drug,
            "activesubstancename"
        )

        drugcharacterization = obtener_texto(
            drug,
            "drugcharacterization"
        )

        drugindication = obtener_texto(
            drug,
            "drugindication"
        )

        registros_medicamentos.append(
            {
                "safetyreportid": safetyreportid,
                "drug_n": drug_n,
                "drugcharacterization": drugcharacterization,
                "medicinalproduct": medicinalproduct,
                "activesubstancename": activesubstancename,
                "drugindication": drugindication,
            }
        )

        if activesubstancename is not None:
            sustancias_activas.append(
                activesubstancename
            )


    # REACCIONES
    terminos_reaccion = []

    for reaction_n, reaction in enumerate(
        reactions,
        start=1
    ):

        reaction_pt = obtener_texto(
            reaction,
            "reactionmeddrapt"
        )

        reaction_version = obtener_texto(
            reaction,
            "reactionmeddraversionpt"
        )

        reaction_outcome = obtener_texto(
            reaction,
            "reactionoutcome"
        )

        registros_reacciones.append(
            {
                "safetyreportid": safetyreportid,
                "reaction_n": reaction_n,
                "reactionmeddrapt": reaction_pt,
                "reactionmeddraversionpt": reaction_version,
                "reactionoutcome": reaction_outcome,
            }
        )

        if reaction_pt is not None:
            terminos_reaccion.append(
                reaction_pt
            )


    # RESUMEN DEL REPORTE

    n_drug = len(drugs)

    n_active_unique = len(
        set(sustancias_activas)
    )

    n_reaction = len(reactions)

    n_reaction_unique = len(
        set(terminos_reaccion)
    )


    registros_reportes.append(
        {
            "safetyreportid":
                safetyreportid,

            "safetyreportversion":
                safetyreportversion,

            "receivedate":
                obtener_texto(elem, "receivedate"),

            "receiptdate":
                obtener_texto(elem, "receiptdate"),

            "occurcountry":
                obtener_texto(elem, "occurcountry"),

            "n_drug":
                n_drug,

            "n_active_unique":
                n_active_unique,

            "n_drug_repetidos":
                n_drug - n_active_unique,

            "n_reaction":
                n_reaction,

            "n_reaction_unique":
                n_reaction_unique,
        }
    )


    # Contabilizar reporte procesado

    contador_reportes += 1


    # Liberar memoria del reporte ya procesado

    elem.clear()


    # Detenerse al alcanzar N_MUESTRA

    if contador_reportes >= N_MUESTRA:
        break


# Construir DataFrames
df_muestra_reportes = pd.DataFrame(
    registros_reportes
)

df_muestra_medicamentos = pd.DataFrame(
    registros_medicamentos
)

df_muestra_reacciones = pd.DataFrame(
    registros_reacciones
)


# Resultado básico
print(
    f"Reportes procesados: "
    f"{len(df_muestra_reportes):,}"
)

print(
    f"Registros <drug> extraídos: "
    f"{len(df_muestra_medicamentos):,}"
)

print(
    f"Registros <reaction> extraídos: "
    f"{len(df_muestra_reacciones):,}"
)

Reportes procesados: 1,000
Registros <drug> extraídos: 6,104
Registros <reaction> extraídos: 3,427


La extracción funcionó correctamente. A partir de $1{,}000$ reportes obtuvimos $6{,}104$ registros `<drug>` y $3{,}427$ registros `<reaction>`, lo que confirma que la estructura uno-a-muchos que observamos en el primer caso es habitual y no una particularidad aislada. Esto es coherente con la documentación de FAERS: un reporte puede contener uno o más medicamentos y uno o más términos MedDRA de reacción. 

Ahora conviene no tocar todavía las fechas. Primero caractericemos bien la multiplicidad, repetición, roles y faltantes de medicamentos/reacciones en estos $1{,}000$ reportes. Eso nos permitirá decidir después cómo deduplicar sin destruir información útil.



## 8. Caracterización de medicamentos y reacciones en una muestra de 1,000 reportes

La extracción incremental de los primeros $1{,}000$ reportes de
`1_ADR25Q1.xml` produjo:

- $1{,}000$ reportes;
- $6{,}104$ registros `<drug>`;
- $3{,}427$ registros `<reaction>`.

Estos resultados confirman que la estructura de FAERS es relacional y que un mismo
reporte puede contener múltiples medicamentos y múltiples reacciones adversas.

Antes de construir pares medicamento--evento es necesario estudiar con mayor detalle
esta multiplicidad.



### 8.1 Número de medicamentos por reporte

Para cada reporte $i$ se definió

$$
m_i=
\text{número de elementos `<drug>` presentes en el reporte}.
$$

También se calculó

$$
m_i^{*}
=
\text{número de sustancias activas distintas dentro del reporte}.
$$

Cuando

$$
m_i > m_i^{*},
$$

al menos una sustancia activa aparece más de una vez dentro del mismo reporte.

Definimos entonces

$$
d_i=m_i-m_i^{*},
$$

donde $d_i$ representa el número de registros adicionales atribuibles a
repeticiones de sustancias activas.

Es importante enfatizar que, en esta etapa, una repetición no será considerada
automáticamente un error. Puede deberse a diferencias en indicación, dosis, vía de
administración u otros elementos del registro original.


### 8.2 Número de reacciones por reporte

De manera análoga, definimos

$$
n_i=
\text{número de elementos `<reaction>` del reporte},
$$

y

$$
n_i^{*}
=
\text{número de términos MedDRA distintos}.
$$

Si

$$
n_i > n_i^{*},
$$

algún término de reacción aparece repetido dentro del mismo reporte.

Esta situación deberá estudiarse antes de construir los pares
medicamento--evento, porque para el análisis de desproporcionalidad un mismo
reporte no debe aumentar artificialmente el conteo de una combinación debido a
registros repetidos del mismo término.



### 8.3 Papel de los medicamentos

También analizaremos la distribución de `drugcharacterization`.

Los códigos considerados inicialmente son:

| Código | Papel del medicamento |
|---|---|
| 1 | Suspect |
| 2 | Concomitant |
| 3 | Interacting |
| 4 | Drug not administered |

Esta distribución permitirá conocer qué proporción de los registros corresponde a
medicamentos sospechosos y qué proporción corresponde a medicamentos con otros
papeles dentro del reporte.

Para el análisis principal de señales se utilizarán posteriormente los medicamentos
clasificados como `Suspect`, pero todavía no se aplicará ese filtro.


### 8.4 Información faltante

Se estudiará además la presencia de valores faltantes en:

- `medicinalproduct`;
- `activesubstancename`;
- `drugindication`;
- `drugcharacterization`;
- `reactionmeddrapt`;
- `reactionmeddraversionpt`;
- `reactionoutcome`.

La proporción de valores faltantes para una variable $X$ se calculará como

$$
p_{\mathrm{miss}}(X)
=
\frac{N_{\mathrm{miss}}(X)}
{N}
\times 100.
$$

Conocer esta información será esencial para decidir posteriormente cómo construir
la variable analítica `drug_key`.



### 8.5 Producto medicinal y sustancia activa

Finalmente, compararemos `medicinalproduct` con `activesubstancename`.

En el primer reporte ambas variables coincidieron para las nueve sustancias
observadas. Sin embargo, no debemos asumir que esto ocurre en todos los reportes.

Por ahora la comparación será estrictamente textual y se realizará sobre los datos
originales, sin aplicar todavía estandarización de mayúsculas, espacios, sales,
nombres comerciales o combinaciones de principios activos.


### Objetivo de esta sección

Al finalizar esta etapa podremos responder:

1. ¿cuántos medicamentos contiene típicamente un reporte?;
2. ¿con qué frecuencia existen sustancias activas repetidas?;
3. ¿cuántas reacciones contiene típicamente un reporte?;
4. ¿con qué frecuencia se repiten términos MedDRA?;
5. ¿cuál es la distribución de `drugcharacterization`?;
6. ¿qué variables presentan mayor cantidad de información faltante?;
7. ¿con qué frecuencia difieren `medicinalproduct` y `activesubstancename`?

Estas respuestas serán utilizadas posteriormente para definir las reglas de
limpieza y deduplicación.

In [23]:
# 8. Caracterización de medicamentos y reacciones en la muestra de 1,000 reportes


# 8.1 Multiplicidad de medicamentos y reacciones por reporte

resumen_multiplicidad = (
    df_muestra_reportes[
        [
            "n_drug",
            "n_active_unique",
            "n_drug_repetidos",
            "n_reaction",
            "n_reaction_unique"
        ]
    ]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
    .T
)

print("RESUMEN DE MULTIPLICIDAD POR REPORTE")
display(resumen_multiplicidad)


# 8.2 Reportes con sustancias activas repetidas
n_reportes = len(df_muestra_reportes)

n_con_drug_repetido = (
    df_muestra_reportes["n_drug_repetidos"] > 0
).sum()

pct_con_drug_repetido = (
    100 * n_con_drug_repetido / n_reportes
)


print("\nREPETICIÓN DE SUSTANCIAS ACTIVAS")

print(
    f"Reportes con al menos una sustancia activa repetida: "
    f"{n_con_drug_repetido:,} de {n_reportes:,} "
    f"({pct_con_drug_repetido:.2f}%)"
)


# 8.3 Reportes con términos de reacción repetidos
mask_reaction_repetida = (
    df_muestra_reportes["n_reaction"]
    >
    df_muestra_reportes["n_reaction_unique"]
)

n_con_reaction_repetida = mask_reaction_repetida.sum()

pct_con_reaction_repetida = (
    100 * n_con_reaction_repetida / n_reportes
)


print("\nREPETICIÓN DE TÉRMINOS DE REACCIÓN")

print(
    f"Reportes con al menos un término MedDRA repetido: "
    f"{n_con_reaction_repetida:,} de {n_reportes:,} "
    f"({pct_con_reaction_repetida:.2f}%)"
)


# 8.4 Distribución de drugcharacterization

mapa_drugcharacterization = {
    "1": "Suspect",
    "2": "Concomitant",
    "3": "Interacting",
    "4": "Drug not administered"
}

df_muestra_medicamentos["drug_role"] = (
    df_muestra_medicamentos["drugcharacterization"]
    .map(mapa_drugcharacterization)
    .fillna("Unknown / other")
)


tabla_roles = (
    df_muestra_medicamentos["drug_role"]
    .value_counts(dropna=False)
    .rename_axis("drug_role")
    .reset_index(name="n_registros")
)

tabla_roles["porcentaje"] = (
    100
    * tabla_roles["n_registros"]
    / len(df_muestra_medicamentos)
)


print("\nDISTRIBUCIÓN DE drugcharacterization")
display(tabla_roles)


# 8.5 Valores faltantes en medicamentos

columnas_drug = [
    "drugcharacterization",
    "medicinalproduct",
    "activesubstancename",
    "drugindication"
]


faltantes_drug = pd.DataFrame({
    "variable": columnas_drug,
    "n_faltantes": [
        df_muestra_medicamentos[col].isna().sum()
        for col in columnas_drug
    ]
})

faltantes_drug["porcentaje_faltante"] = (
    100
    * faltantes_drug["n_faltantes"]
    / len(df_muestra_medicamentos)
)


print("\nVALORES FALTANTES EN MEDICAMENTOS")
display(faltantes_drug)


# 8.6 Valores faltantes en reacciones

columnas_reaction = [
    "reactionmeddrapt",
    "reactionmeddraversionpt",
    "reactionoutcome"
]


faltantes_reaction = pd.DataFrame({
    "variable": columnas_reaction,
    "n_faltantes": [
        df_muestra_reacciones[col].isna().sum()
        for col in columnas_reaction
    ]
})

faltantes_reaction["porcentaje_faltante"] = (
    100
    * faltantes_reaction["n_faltantes"]
    / len(df_muestra_reacciones)
)


print("\nVALORES FALTANTES EN REACCIONES")
display(faltantes_reaction)


# 8.7 Comparación medicinalproduct vs activesubstancename

comparacion_nombres = (
    df_muestra_medicamentos[
        ["medicinalproduct", "activesubstancename"]
    ]
    .dropna()
    .copy()
)


comparacion_nombres["coinciden_exactamente"] = (
    comparacion_nombres["medicinalproduct"]
    ==
    comparacion_nombres["activesubstancename"]
)


n_comparables = len(comparacion_nombres)

n_coinciden = (
    comparacion_nombres["coinciden_exactamente"].sum()
)

n_difieren = (
    (~comparacion_nombres["coinciden_exactamente"]).sum()
)


print(
    "\nCOMPARACIÓN ENTRE medicinalproduct "
    "Y activesubstancename"
)

print(
    f"Registros comparables: {n_comparables:,}"
)

print(
    f"Coinciden exactamente: "
    f"{n_coinciden:,} "
    f"({100*n_coinciden/n_comparables:.2f}%)"
)

print(
    f"Difieren: "
    f"{n_difieren:,} "
    f"({100*n_difieren/n_comparables:.2f}%)"
)


# 8.8 Ejemplos donde los nombres son diferentes

ejemplos_nombres_diferentes = (
    comparacion_nombres[
        ~comparacion_nombres["coinciden_exactamente"]
    ][
        [
            "medicinalproduct",
            "activesubstancename"
        ]
    ]
    .drop_duplicates()
    .head(20)
    .reset_index(drop=True)
)


print(
    "\nPRIMEROS EJEMPLOS DONDE "
    "medicinalproduct != activesubstancename"
)

display(ejemplos_nombres_diferentes)


# 8.9 Versiones de MedDRA observadas

versiones_meddra = (
    df_muestra_reacciones[
        "reactionmeddraversionpt"
    ]
    .value_counts(dropna=False)
    .rename_axis("version_MedDRA")
    .reset_index(name="n_reacciones")
)


print("\nVERSIONES DE MedDRA OBSERVADAS")
display(versiones_meddra)

RESUMEN DE MULTIPLICIDAD POR REPORTE


,count,mean,std,min,25%,50%,75%,90%,95%,99%,max
n_drug,1000.0,6.104,25.067340,1.0,1.0,2.0,5.0,12.0,18.00,44.01,505.0
n_active_unique,1000.0,3.586,6.555271,1.0,1.0,1.0,3.0,9.0,13.05,27.01,87.0
n_drug_repetidos,1000.0,2.518,20.449887,0.0,0.0,0.0,1.0,3.0,6.00,13.12,419.0
n_reaction,1000.0,3.427,7.246046,1.0,1.0,2.0,3.0,6.0,10.00,23.00,170.0
n_reaction_unique,1000.0,3.396,7.023711,1.0,1.0,2.0,3.0,6.0,10.00,23.00,161.0



REPETICIÓN DE SUSTANCIAS ACTIVAS
Reportes con al menos una sustancia activa repetida: 387 de 1,000 (38.70%)

REPETICIÓN DE TÉRMINOS DE REACCIÓN
Reportes con al menos un término MedDRA repetido: 12 de 1,000 (1.20%)

DISTRIBUCIÓN DE drugcharacterization


,drug_role,n_registros,porcentaje
0,Concomitant,3143,51.490826
1,Suspect,2938,48.132372
2,Interacting,23,0.376802



VALORES FALTANTES EN MEDICAMENTOS


...

,variable,n_faltantes,porcentaje_faltante
0,reactionmeddrapt,0,0.0
1,reactionmeddraversionpt,0,0.0
2,reactionoutcome,0,0.0



COMPARACIÓN ENTRE medicinalproduct Y activesubstancename
Registros comparables: 6,034
Coinciden exactamente: 3,729 (61.80%)
Difieren: 2,305 (38.20%)

PRIMEROS EJEMPLOS DONDE medicinalproduct != activesubstancename


,medicinalproduct,activesubstancename
0,VITAMIN K,PHYTONADIONE
1,CIMZIA,CERTOLIZUMAB PEGOL
2,SYMBICORT,BUDESONIDE\FORMOTEROL FUMARATE DIHYDRATE
3,DUPIXENT,DUPILUMAB
4,CRESTOR,ROSUVASTATIN CALCIUM
5,VYNDAMAX,TAFAMIDIS
6,AMOXIL,AMOXICILLIN
7,CLOZARIL,CLOZAPINE
8,METROGEL,METRONIDAZOLE
9,NIZORAL,KETOCONAZOLE



VERSIONES DE MedDRA OBSERVADAS


,version_MedDRA,n_reacciones
0,27.1,3427


Estos resultados nos permiten tomar una decisión metodológica importante.

La variable `activesubstancename` parece ser la mejor candidata para identificar analíticamente al medicamento porque evita mezclar marcas comerciales con principios activos. La diferencia entre `medicinalproduct` y `activesubstancename` es sustancial: en la muestra difieren en **38.20%** de los registros comparables. Ejemplos como `CIMZIA → CERTOLIZUMAB PEGOL`, `DUPIXENT → DUPILUMAB` o `REVLIMID → LENALIDOMIDE` muestran por qué no conviene utilizar directamente `medicinalproduct` como identificador principal.

Esto es coherente con la documentación de FAERS: la FDA incorporó `<activesubstancename>` específicamente para contener el ingrediente activo del producto, y advierte que los valores de `medicinalproduct` pueden presentar cambios derivados de sus diccionarios de medicamentos.  

Pero tampoco podemos usar únicamente `activesubstancename`, porque está ausente en **70 de 6,104 registros, es decir, 1.15%**. Por eso propongo una regla jerárquica provisional:

$$
\texttt{drug_key_raw}=
\begin{cases}
\texttt{activesubstancename}, & \text{si está disponible},\\
\texttt{medicinalproduct}, & \text{si falta la sustancia activa}.
\end{cases}
$$

Antes de normalizar textos o eliminar duplicados, debemos comprobar cómo se comporta esta regla en la muestra.

También hay dos resultados que vale la pena resaltar. El **38.7% de los reportes tiene al menos una sustancia activa repetida**, así que la deduplicación no será un detalle menor. En cambio, las reacciones repetidas son poco frecuentes, solo **1.2%**, aunque de todas formas deberemos deduplicarlas a nivel reporte–término MedDRA.

Además, `drugcharacterization` está prácticamente dividido en dos grandes grupos: 48.13% `Suspect` y 51.49% `Concomitant`. Esto confirma que sería metodológicamente incorrecto usar todos los medicamentos para generar señales: los concomitantes formarían aproximadamente la mitad de los registros.

## 9. Construcción provisional de un identificador analítico del medicamento

La exploración de los primeros $1{,}000$ reportes mostró que las variables
`medicinalproduct` y `activesubstancename` no son equivalentes.

De los registros en los que ambas variables estuvieron disponibles, solamente el

$$
61.80\%
$$

presentó una coincidencia exacta entre ambas.

En el

$$
38.20\%
$$

restante se observaron diferencias como:

- nombre comercial frente a principio activo;
- sal farmacéutica frente a denominación del producto;
- productos con múltiples principios activos;
- otras diferencias en la forma en que el medicamento fue reportado.

Por ejemplo:

    CIMZIA     -> CERTOLIZUMAB PEGOL
    DUPIXENT   -> DUPILUMAB
    REVLIMID   -> LENALIDOMIDE
    NOVOLOG    -> INSULIN ASPART

Para un análisis de desproporcionalidad es preferible trabajar, en la medida de lo
posible, con la **sustancia activa**, porque distintos nombres comerciales pueden
representar el mismo principio farmacológico.

Por esta razón se construirá inicialmente una variable denominada

`drug_key_raw`.

La regla provisional será

$$
\texttt{drug\_key\_raw}
=
\begin{cases}
\texttt{activesubstancename},
&
\text{si la sustancia activa está disponible},
\\[6pt]
\texttt{medicinalproduct},
&
\text{si la sustancia activa está ausente}.
\end{cases}
$$

El término `raw` indica que todavía no se realizará ninguna normalización del texto.

Por ejemplo, en esta etapa todavía no se modificarán:

- mayúsculas y minúsculas;
- espacios adicionales;
- signos de puntuación;
- sales farmacéuticas;
- combinaciones de ingredientes;
- nombres equivalentes;
- denominaciones comerciales.

Estas transformaciones deberán estudiarse posteriormente y no deben imponerse sin
examinar primero los datos.


### 9.1 ¿Por qué utilizar una regla jerárquica?

En la muestra, `activesubstancename` presentó aproximadamente un

$$
1.15\%
$$

de valores faltantes.

Eliminar automáticamente esos registros implicaría perder información que podría ser
recuperable mediante `medicinalproduct`.

Por ello, el uso de una regla jerárquica permite maximizar la disponibilidad del
identificador del medicamento sin sustituir indiscriminadamente la información
original.


### 9.2 Cohorte de medicamentos sospechosos

Para el análisis principal de señales no se utilizarán todos los medicamentos
registrados en un caso.

En la muestra estudiada, aproximadamente la mitad de los registros correspondió a
medicamentos concomitantes.

Por tanto, la cohorte principal se definirá inicialmente mediante

$$
\texttt{drugcharacterization}=1,
$$

es decir, medicamentos clasificados como `Suspect`.

Definiremos

$$
\mathcal{D}_{S}
=
\{
d:
\texttt{drugcharacterization}(d)=1
\}.
$$

Todavía no eliminaremos los medicamentos concomitantes de las tablas originales.
En su lugar, construiremos una tabla analítica separada con los medicamentos
sospechosos.

Esto permitirá conservar los datos originales y mantener trazabilidad durante todo
el procesamiento.



### 9.3 Deduplicación a nivel reporte--medicamento

La muestra mostró que el

$$
38.7\%
$$

de los reportes contiene al menos una sustancia activa repetida.

Por lo tanto, antes de calcular medidas de desproporcionalidad será necesario evitar
que una misma sustancia contribuya varias veces dentro del mismo reporte.

La unidad analítica propuesta para los medicamentos será

$$
(
\texttt{safetyreportid},
\texttt{drug\_key\_raw}
).
$$

Después de aplicar el filtro de medicamentos sospechosos, cada combinación

$$
(\text{reporte},\text{medicamento})
$$

deberá aparecer como máximo una vez.

Es importante señalar que esta deduplicación no significa que los registros
repetidos originales sean errores. Los diferentes elementos `<drug>` pueden contener
información distinta sobre indicaciones, dosis, rutas u otras características.

La deduplicación se realizará únicamente para construir la tabla utilizada en el
análisis de señales.


### 9.4 Objetivos de esta sección

En esta etapa se evaluará:

1. cuántos registros reciben un `drug_key_raw`;
2. cuántos requieren utilizar `medicinalproduct` como respaldo;
3. cuántos medicamentos sospechosos existen antes de deduplicar;
4. cuántos pares únicos reporte--medicamento permanecen después de deduplicar;
5. cuántos registros repetidos se eliminan de la tabla analítica;
6. cuáles son los medicamentos sospechosos más frecuentes en la muestra.

Todavía no se modificarán los nombres ni se construirán pares con las reacciones.

In [25]:
# 9. Construcción provisional de drug_key_raw y cohorte de medicamentos Suspect


# 9.1 Construir drug_key_raw
df_muestra_medicamentos["drug_key_raw"] = (
    df_muestra_medicamentos["activesubstancename"]
    .fillna(
        df_muestra_medicamentos["medicinalproduct"]
    )
)


# Identificar de dónde provino el identificador
df_muestra_medicamentos["drug_key_source"] = (
    df_muestra_medicamentos["activesubstancename"]
    .notna()
    .map({
        True: "activesubstancename",
        False: "medicinalproduct_fallback"
    })
)


# 9.2 Disponibilidad del identificador

n_total_drug = len(df_muestra_medicamentos)

n_key_disponible = (
    df_muestra_medicamentos["drug_key_raw"]
    .notna()
    .sum()
)

n_key_faltante = (
    df_muestra_medicamentos["drug_key_raw"]
    .isna()
    .sum()
)


print("DISPONIBILIDAD DE drug_key_raw")

print(
    f"Registros <drug>: {n_total_drug:,}"
)

print(
    f"Con drug_key_raw disponible: "
    f"{n_key_disponible:,} "
    f"({100*n_key_disponible/n_total_drug:.2f}%)"
)

print(
    f"Sin drug_key_raw: "
    f"{n_key_faltante:,} "
    f"({100*n_key_faltante/n_total_drug:.2f}%)"
)


# 9.3 Fuente utilizada para drug_key_raw

tabla_fuente_key = (
    df_muestra_medicamentos["drug_key_source"]
    .value_counts(dropna=False)
    .rename_axis("fuente")
    .reset_index(name="n_registros")
)

tabla_fuente_key["porcentaje"] = (
    100
    * tabla_fuente_key["n_registros"]
    / n_total_drug
)


print("\nFUENTE DE drug_key_raw")
display(tabla_fuente_key)


# 9.4 Examinar los casos que requirieron fallback

fallback_ejemplos = (
    df_muestra_medicamentos[
        df_muestra_medicamentos[
            "drug_key_source"
        ] == "medicinalproduct_fallback"
    ][
        [
            "safetyreportid",
            "drugcharacterization",
            "medicinalproduct",
            "activesubstancename",
            "drugindication",
            "drug_key_raw"
        ]
    ]
    .drop_duplicates()
    .head(20)
    .reset_index(drop=True)
)


print(
    "\nPRIMEROS EJEMPLOS QUE UTILIZARON "
    "medicinalproduct COMO FALLBACK"
)

display(fallback_ejemplos)


# 9.5 Construir cohorte Suspect
df_suspect_raw = (
    df_muestra_medicamentos[
        df_muestra_medicamentos[
            "drugcharacterization"
        ] == "1"
    ]
    .copy()
)


print("\nCOHORTE SUSPECT ANTES DE DEDUPLICAR")

print(
    f"Registros Suspect: "
    f"{len(df_suspect_raw):,}"
)

print(
    f"Reportes con al menos un medicamento Suspect: "
    f"{df_suspect_raw['safetyreportid'].nunique():,}"
)

print(
    f"drug_key_raw distintos: "
    f"{df_suspect_raw['drug_key_raw'].nunique(dropna=True):,}"
)


# 9.6 Detectar duplicados reporte-medicamento
duplicados_suspect = (
    df_suspect_raw
    .duplicated(
        subset=[
            "safetyreportid",
            "drug_key_raw"
        ],
        keep=False
    )
)


n_registros_en_grupos_duplicados = (
    duplicados_suspect.sum()
)


print(
    "\nDUPLICACIÓN DENTRO DE LA COHORTE SUSPECT"
)

print(
    f"Registros pertenecientes a combinaciones "
    f"reporte-medicamento repetidas: "
    f"{n_registros_en_grupos_duplicados:,}"
)



# 9.7 Deduplicar únicamente para la tabla analítica
df_suspect_unico = (
    df_suspect_raw
    .dropna(
        subset=["drug_key_raw"]
    )
    .drop_duplicates(
        subset=[
            "safetyreportid",
            "drug_key_raw"
        ]
    )
    .reset_index(drop=True)
)


n_antes = len(df_suspect_raw)
n_despues = len(df_suspect_unico)

n_eliminados = n_antes - n_despues


print("\nDEDUPLICACIÓN ANALÍTICA")

print(
    f"Registros Suspect antes: "
    f"{n_antes:,}"
)

print(
    f"Pares únicos reporte-medicamento después: "
    f"{n_despues:,}"
)

print(
    f"Registros redundantes para el análisis: "
    f"{n_eliminados:,}"
)

print(
    f"Reducción relativa: "
    f"{100*n_eliminados/n_antes:.2f}%"
)


# 9.8 Número de medicamentos Suspect únicos por reporte
suspect_por_reporte = (
    df_suspect_unico
    .groupby("safetyreportid")
    .size()
    .rename("n_suspect_unicos")
    .reset_index()
)


print(
    "\nNÚMERO DE MEDICAMENTOS SUSPECT ÚNICOS POR REPORTE"
)

display(
    suspect_por_reporte[
        "n_suspect_unicos"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 9.9 Medicamentos Suspect más frecuentes

top_suspect = (
    df_suspect_unico[
        "drug_key_raw"
    ]
    .value_counts()
    .head(20)
    .rename_axis("drug_key_raw")
    .reset_index(name="n_reportes")
)


print(
    "\nTOP 20 DE MEDICAMENTOS SUSPECT "
    "EN LA MUESTRA"
)

display(top_suspect)

DISPONIBILIDAD DE drug_key_raw
Registros <drug>: 6,104
Con drug_key_raw disponible: 6,104 (100.00%)
Sin drug_key_raw: 0 (0.00%)

FUENTE DE drug_key_raw


,fuente,n_registros,porcentaje
0,activesubstancename,6034,98.853211
1,medicinalproduct_fallback,70,1.146789



PRIMEROS EJEMPLOS QUE UTILIZARON medicinalproduct COMO FALLBACK


,safetyreportid,drugcharacterization,medicinalproduct,activesubstancename,drugindication,drug_key_raw
0,24726962,2,COVID-19 vaccine,NaN,COVID-19 prophylaxis,COVID-19 vaccine
1,24192176,2,DULCOLAX LIQUID,NaN,Constipation,DULCOLAX LIQUID
2,21652785,2,MOUTHWASHGUM,NaN,Product used for unknown indication,MOUTHWASHGUM
3,24795669,2,CHILDRENS CHEWABLE VITAMINS,NaN,Product used for unknown indication,CHILDRENS CHEWABLE VITAMINS
4,24741426,2,TREZEN,NaN,Product used for unknown indication,TREZEN
5,24379964,2,VITAMIN D3 K2,NaN,NaN,VITAMIN D3 K2
6,24795768,2,AIMOVIG INI,NaN,NaN,AIMOVIG INI
7,24795768,2,UPTRAVI (MNTH 1 TITR),NaN,NaN,UPTRAVI (MNTH 1 TITR)
8,24795768,2,TYVASO DPI TITRAT KIT POW,NaN,NaN,TYVASO DPI TITRAT KIT POW
9,24222173,2,BIOFERMIN [BIFIDOBACTERIUM NOS],NaN,NaN,BIOFERMIN [BIFIDOBACTERIUM NOS]



COHORTE SUSPECT ANTES DE DEDUPLICAR
Registros Suspect: 2,938
Reportes con al menos un medicamento Suspect: 995
drug_key_raw distintos: 538

DUPLICACIÓN DENTRO DE LA COHORTE SUSPECT
Registros pertenecientes a combinaciones reporte-medicamento repetidas: 1,856

DEDUPLICACIÓN ANALÍTICA
Registros Suspect antes: 2,938
Pares únicos reporte-medicamento después: 1,589
Registros redundantes para el análisis: 1,349
Reducción relativa: 45.92%

NÚMERO DE MEDICAMENTOS SUSPECT ÚNICOS POR REPORTE


,valor
count,995.000000
mean,1.596985
std,2.504200
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
90%,2.600000
95%,4.000000
99%,9.060000



TOP 20 DE MEDICAMENTOS SUSPECT EN LA MUESTRA


,drug_key_raw,n_reportes
0,DUPILUMAB,107
1,PIMAVANSERIN TARTRATE,44
2,LENALIDOMIDE,29
3,RELUGOLIX,27
4,ABATACEPT,24
5,TOCILIZUMAB,20
6,NIVOLUMAB,18
7,POMALIDOMIDE,18
8,BEVACIZUMAB,18
9,APIXABAN,16


Estos resultados ya justifican una decisión metodológica bastante clara: **la unidad para el análisis no puede ser el elemento `<drug>` original**, sino el par único reporte–medicamento. La reducción de $2{,}938$ registros `Suspect` a $1{,}589$ pares únicos implica que el **45.92%** de las filas `Suspect` son redundantes para el conteo de presencia medicamento–reporte.

Eso no significa que esas filas estén “mal”: pueden representar información distinta asociada al mismo medicamento. La FDA advierte precisamente que un producto puede aparecer en más de una fila dentro de un caso.  Para PRR/ROR, sin embargo, lo que necesitamos es saber si el medicamento estuvo o no presente en el reporte, no cuántas filas XML produjo.

La cobertura de nuestra regla:

$$
\texttt{drug_key_raw}=
\begin{cases}
\texttt{activesubstancename},\
\texttt{medicinalproduct}\text{ como respaldo}.
\end{cases}
$$

 alcanzó **100% de cobertura** en la muestra. La FDA documenta `activesubstancename` como el campo de ingrediente activo, por lo que utilizarlo como primera opción está bien fundamentado. 

## 10. Validación de `drug_key_raw` y análisis de la duplicación en medicamentos sospechosos

La regla provisional utilizada para identificar analíticamente los medicamentos logró
asignar un valor de `drug_key_raw` al $100\%$ de los $6{,}104$ registros de medicamentos de la muestra.

En el $98.85\%$ de los casos se utilizó directamente `activesubstancename`, mientras que en el $1.15\%$ restante fue necesario utilizar `medicinalproduct` como respaldo.

Antes de convertir `drug_key_raw` en el identificador definitivo `drug_key`,
debemos comprobar cómo se comporta específicamente dentro de la cohorte de
medicamentos sospechosos.


### 10.1 Importancia de estudiar por separado la cohorte `Suspect`

El análisis de desproporcionalidad utilizará principalmente medicamentos con

$$
\texttt{drugcharacterization}=1.
$$

Por esta razón, es posible que algunos problemas observados en el conjunto completo
de medicamentos no sean relevantes para la cohorte analítica.

En particular, queremos determinar cuántos medicamentos `Suspect` requieren utilizar

`medicinalproduct`

como sustituto de `activesubstancename`.

Si este número es muy pequeño, la identificación de medicamentos dentro de la
cohorte principal será especialmente robusta.



### 10.2 Multiplicidad de un mismo medicamento dentro de un reporte

La deduplicación de los medicamentos `Suspect` produjo una reducción importante:

$$
2938 \longrightarrow 1589
$$

registros analíticos.

La reducción absoluta fue

$$
2938-1589=1349
$$

registros, correspondiente aproximadamente al

$$
45.92\%.
$$

Ahora estudiaremos cuántas veces aparece repetido un mismo medicamento dentro de un
reporte.

Para cada combinación

$$
(\texttt{safetyreportid},\texttt{drug\_key\_raw})
$$

definiremos su multiplicidad como

$$
r_{ij}
=
\text{número de elementos `<drug>` correspondientes al medicamento }j
\text{ en el reporte }i.
$$

Si

$$
r_{ij}=1,
$$

el medicamento aparece una sola vez.

Si

$$
r_{ij}>1,
$$

existen registros XML repetidos del mismo medicamento dentro del reporte.

Conocer la distribución de $r_{ij}$ permitirá cuantificar la magnitud de este
fenómeno.



### 10.3 Normalización textual mínima

Antes de realizar cualquier estandarización farmacológica se evaluará únicamente una
normalización **conservadora** del texto.

Se construirá temporalmente una versión que:

1. elimine espacios al inicio y al final;
2. convierta el texto a mayúsculas;
3. sustituya secuencias de varios espacios por un único espacio.

Por ejemplo,

$$
\texttt{"  Dupilumab  "}
\longrightarrow
\texttt{"DUPILUMAB"}.
$$

Esta transformación no modifica sales, nombres químicos ni combinaciones de
principios activos.

Por ahora **no se realizarán** transformaciones como

$$
\texttt{PIMAVANSERIN TARTRATE}
\rightarrow
\texttt{PIMAVANSERIN},
$$

porque eliminar una sal farmacéutica representa una decisión de armonización
farmacológica y no una simple limpieza de texto.

Tampoco se separarán automáticamente sustancias combinadas.



### 10.4 Medicamentos con múltiples principios activos

Algunos valores de `activesubstancename` pueden representar más de una sustancia
activa.

Por ejemplo, pueden aparecer separadores como:

- `\`;
- `/`;
- `+`;
- `;`.

Estos casos serán identificados, pero todavía no serán separados.

La decisión futura deberá establecer si una combinación farmacológica será tratada
como una entidad propia

$$
D=A+B,
$$

o si se descompondrá en sus componentes

$$
D=\{A,B\}.
$$

Esta elección puede modificar los conteos de desproporcionalidad y, por tanto, no
debe realizarse automáticamente.



### 10.5 Reportes con un número elevado de medicamentos sospechosos

La muestra mostró que el número de medicamentos `Suspect` únicos por reporte tiene
mediana igual a

$$
1,
$$

pero alcanza un máximo de

$$
56.
$$

Por ello, también se identificarán los reportes con mayor número de medicamentos
sospechosos.

Estos casos no serán eliminados. Se examinarán únicamente para determinar si
corresponden a situaciones plausibles de polifarmacia, tratamientos combinados o a
estructuras particulares de los reportes.



### Objetivo de esta sección

Antes de fijar definitivamente `drug_key`, esta etapa permitirá responder:

1. ¿cuántos medicamentos `Suspect` requieren el respaldo de `medicinalproduct`?;
2. ¿cuál es la distribución de la multiplicidad de los pares reporte--medicamento?;
3. ¿la limpieza mínima de mayúsculas y espacios reduce el número de nombres distintos?;
4. ¿qué variantes textuales se fusionarían mediante esa limpieza?;
5. ¿qué tan frecuentes son las combinaciones de principios activos?;
6. ¿qué reportes contienen cantidades excepcionalmente altas de medicamentos sospechosos?

Todavía no se modificará permanentemente ninguna variable.

In [27]:
# 10. Validación de drug_key_raw y duplicación en la cohorte Suspect

import re


# 10.1 Fuente de drug_key_raw dentro de la cohorte Suspect

tabla_fuente_suspect = (
    df_suspect_raw["drug_key_source"]
    .value_counts(dropna=False)
    .rename_axis("fuente")
    .reset_index(name="n_registros")
)

tabla_fuente_suspect["porcentaje"] = (
    100
    * tabla_fuente_suspect["n_registros"]
    / len(df_suspect_raw)
)

print("FUENTE DE drug_key_raw EN LA COHORTE SUSPECT")
display(tabla_fuente_suspect)


# 10.2 Casos Suspect que utilizaron medicinalproduct como fallback
suspect_fallback = (
    df_suspect_raw[
        df_suspect_raw["drug_key_source"]
        == "medicinalproduct_fallback"
    ]
    .copy()
)

print(
    "\nMEDICAMENTOS SUSPECT QUE REQUIRIERON FALLBACK"
)

print(
    f"Número de registros: {len(suspect_fallback):,}"
)

print(
    f"Número de reportes: "
    f"{suspect_fallback['safetyreportid'].nunique():,}"
)

display(
    suspect_fallback[
        [
            "safetyreportid",
            "medicinalproduct",
            "activesubstancename",
            "drugindication",
            "drug_key_raw"
        ]
    ]
    .drop_duplicates()
    .head(30)
)


# 10.3 Multiplicidad de cada par reporte-medicamento
multiplicidad_suspect = (
    df_suspect_raw
    .groupby(
        [
            "safetyreportid",
            "drug_key_raw"
        ],
        dropna=False
    )
    .size()
    .rename("multiplicidad")
    .reset_index()
)


tabla_multiplicidad = (
    multiplicidad_suspect["multiplicidad"]
    .value_counts()
    .sort_index()
    .rename_axis("multiplicidad")
    .reset_index(name="n_pares")
)

tabla_multiplicidad["porcentaje"] = (
    100
    * tabla_multiplicidad["n_pares"]
    / len(multiplicidad_suspect)
)


print(
    "\nDISTRIBUCIÓN DE LA MULTIPLICIDAD "
    "REPORTE-MEDICAMENTO"
)

display(tabla_multiplicidad)


print(
    "\nRESUMEN DE LA MULTIPLICIDAD"
)

display(
    multiplicidad_suspect[
        "multiplicidad"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 10.4 Casos con mayor multiplicidad del mismo medicamento

print(
    "\nPARES REPORTE-MEDICAMENTO "
    "CON MAYOR MULTIPLICIDAD"
)

display(
    multiplicidad_suspect
    .sort_values(
        "multiplicidad",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)


# 10.5 Función de normalización textual mínima

def normalizar_drug_text_minimo(valor):
    """
    Normalización conservadora:

    - elimina espacios exteriores;
    - convierte a mayúsculas;
    - colapsa espacios consecutivos.

    No modifica sales, puntuación ni combinaciones.
    """

    if pd.isna(valor):
        return None

    valor = str(valor).strip().upper()

    valor = re.sub(
        r"\s+",
        " ",
        valor
    )

    return valor


df_suspect_raw["drug_key_min"] = (
    df_suspect_raw["drug_key_raw"]
    .apply(normalizar_drug_text_minimo)
)


# 10.6 Comparar número de nombres antes y después

n_raw = (
    df_suspect_raw["drug_key_raw"]
    .nunique(dropna=True)
)

n_min = (
    df_suspect_raw["drug_key_min"]
    .nunique(dropna=True)
)


print(
    "\nEFECTO DE LA NORMALIZACIÓN TEXTUAL MÍNIMA"
)

print(
    f"Nombres distintos antes: {n_raw:,}"
)

print(
    f"Nombres distintos después: {n_min:,}"
)

print(
    f"Nombres fusionados: {n_raw - n_min:,}"
)


# 10.7 Identificar variantes que serían fusionadas

variantes_textuales = (
    df_suspect_raw[
        [
            "drug_key_raw",
            "drug_key_min"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "drug_key_min",
        dropna=False
    )["drug_key_raw"]
    .apply(
        lambda x: sorted(
            set(x)
        )
    )
    .reset_index(name="variantes_raw")
)

variantes_textuales["n_variantes"] = (
    variantes_textuales[
        "variantes_raw"
    ].apply(len)
)


variantes_fusionadas = (
    variantes_textuales[
        variantes_textuales["n_variantes"] > 1
    ]
    .sort_values(
        "n_variantes",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\nVARIANTES TEXTUALES QUE SERÍAN "
    "FUSIONADAS POR LA LIMPIEZA MÍNIMA"
)

display(
    variantes_fusionadas.head(30)
)


# 10.8 Identificar posibles combinaciones de principios activos

patron_combinacion = r"[\\/+;]"

mask_combinacion = (
    df_suspect_raw["drug_key_raw"]
    .fillna("")
    .str.contains(
        patron_combinacion,
        regex=True
    )
)


combinaciones_suspect = (
    df_suspect_raw.loc[
        mask_combinacion,
        "drug_key_raw"
    ]
    .value_counts()
    .rename_axis("drug_key_raw")
    .reset_index(name="n_registros")
)


print(
    "\nPOSIBLES COMBINACIONES "
    "DE PRINCIPIOS ACTIVOS"
)

print(
    f"Registros Suspect con algún separador "
    f"de combinación: {mask_combinacion.sum():,}"
)

print(
    f"Nombres distintos detectados: "
    f"{combinaciones_suspect['drug_key_raw'].nunique():,}"
)

display(
    combinaciones_suspect.head(30)
)


# 10.9 Reportes con mayor número de medicamentos Suspect únicos

top_reportes_suspect = (
    suspect_por_reporte
    .sort_values(
        "n_suspect_unicos",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)


print(
    "\nREPORTES CON MAYOR NÚMERO "
    "DE MEDICAMENTOS SUSPECT ÚNICOS"
)

display(top_reportes_suspect)

FUENTE DE drug_key_raw EN LA COHORTE SUSPECT


,fuente,n_registros,porcentaje
0,activesubstancename,2938,100.0



MEDICAMENTOS SUSPECT QUE REQUIRIERON FALLBACK
Número de registros: 0
Número de reportes: 0


,safetyreportid,medicinalproduct,activesubstancename,drugindication,drug_key_raw



DISTRIBUCIÓN DE LA MULTIPLICIDAD REPORTE-MEDICAMENTO


...

,drug_key_min,variantes_raw,n_variantes



POSIBLES COMBINACIONES DE PRINCIPIOS ACTIVOS
Registros Suspect con algún separador de combinación: 119
Nombres distintos detectados: 39


,drug_key_raw,n_registros
0,BUDESONIDE\FORMOTEROL\GLYCOPYRROLATE,17
1,BUDESONIDE\FORMOTEROL FUMARATE DIHYDRATE,16
2,FLUTICASONE FUROATE\UMECLIDINIUM BROMIDE\VILAN...,10
3,FORMOTEROL FUMARATE DIHYDRATE\MOMETASONE FUROATE,9
4,CABOTEGRAVIR\RILPIVIRINE,6
5,LEUCOVORIN\LEUCOVORIN CALCIUM,6
6,SACUBITRIL\VALSARTAN,5
7,HUMAN IMMUNOGLOBULIN G\HYALURONIDASE (HUMAN RE...,5
8,TREMELIMUMAB\TREMELIMUMAB-ACTL,4
9,AMPHETAMINE ASPARTATE\AMPHETAMINE SULFATE\DEXT...,4



REPORTES CON MAYOR NÚMERO DE MEDICAMENTOS SUSPECT ÚNICOS


,safetyreportid,n_suspect_unicos
0,24796413,56
1,15551360,28
2,24795713,19
3,24789549,18
4,19178544,14
5,24757336,13
6,15688545,12
7,24795726,12
8,24796067,11
9,24796398,10


Estos resultados son bastante concluyentes, nos muestran que conviene hacer una última auditoría antes de fijar definitivamente `drug_key` y la regla de deduplicación.

En la muestra de $1{,}000$ reportes, los $2{,}938$ registros clasificados como `Suspect` tienen `activesubstancename` disponible en el 100% de los casos. Esto es excelente: al menos para la cohorte que realmente usaremos en el análisis de señales, no necesitamos recurrir a `medicinalproduct` como respaldo. Además, la normalización textual mínima no fusionó ningún nombre, lo que sugiere que los nombres de sustancia activa vienen bastante estandarizados en esta muestra.

Por otra parte, la duplicación es sustancial. De los $1{,}589$ pares únicos reporte–medicamento, alrededor del $31.9%$ aparece más de una vez, y existen casos extremos como `TOCILIZUMAB` apareciendo 85 veces dentro del mismo reporte. La FDA advierte que un producto puede tener múltiples registros dentro de un caso debido a información repetida o múltiple asociada al medicamento, por ejemplo dosis, ruta o frecuencia.  Por eso, antes de asumir que las 85 filas son simplemente redundantes, debemos entender qué cambia entre ellas.

También detectamos $39$ nombres de sustancias sospechosas que representan combinaciones de principios activos. Por ahora las mantendremos como una sola entidad farmacológica, exactamente como aparecen en `activesubstancename`. Separarlas ahora produciría exposiciones a componentes individuales que no están necesariamente representadas así en el reporte original. Podemos reconsiderarlo más adelante como análisis de sensibilidad.

## 11. Auditoría de reportes con multiplicidad elevada de medicamentos sospechosos

El análisis anterior mostró que la deduplicación a nivel

$$
(\texttt{safetyreportid},\texttt{drug\_key\_raw})
$$

produce una reducción importante del número de registros de medicamentos
sospechosos.

En la muestra de $1{,}000$ reportes se observaron $1{,}589$ combinaciones únicas
reporte--medicamento. De ellas, aproximadamente el $68\%$ apareció una sola vez,
mientras que el resto presentó algún grado de repetición.

La mayoría de las multiplicidades fueron pequeñas. Sin embargo, se identificaron
algunos casos extremos.

Por ejemplo, la combinación

$$
(\texttt{safetyreportid}=9383196,\;
\texttt{TOCILIZUMAB})
$$

apareció 85 veces dentro del mismo reporte.

También se observó que el reporte

$$
\texttt{safetyreportid}=15551360
$$

contiene numerosos medicamentos que aparecen repetidamente.

Estos valores no deben interpretarse automáticamente como errores.



### 11.1 ¿Por qué puede repetirse un medicamento?

Un elemento `<drug>` representa un registro de información asociada con un
medicamento dentro del reporte.

Un mismo medicamento puede aparecer varias veces cuando cambia alguna característica,
por ejemplo:

- indicación;
- dosis;
- unidad de dosis;
- vía de administración;
- duración del tratamiento;
- acción tomada con el medicamento;
- otra información asociada con la exposición.

Por tanto, dos elementos `<drug>` pueden representar la misma sustancia activa pero
contener información clínica o administrativa diferente.

Para un análisis descriptivo detallado estas diferencias pueden ser importantes.

Sin embargo, para un análisis de desproporcionalidad basado en presencia o ausencia
del medicamento dentro de un reporte, contar todas estas filas de manera independiente
produciría una sobreponderación artificial del caso.



### 11.2 Presencia frente a número de registros

El análisis de desproporcionalidad utilizará posteriormente una variable binaria
conceptual:

$$
X_{iD}
=
\begin{cases}
1, & \text{si el medicamento }D\text{ aparece en el reporte }i,\\
0, & \text{en otro caso}.
\end{cases}
$$

En este contexto, si `TOCILIZUMAB` aparece $85$ veces dentro del mismo reporte,
la contribución del reporte al conteo del medicamento deberá seguir siendo

$$
X_{i,\text{TOCILIZUMAB}}=1,
$$

y no $85$.

No obstante, antes de aplicar esta regla al conjunto completo, verificaremos qué
características cambian entre los registros repetidos.



### 11.3 Estrategia de auditoría

Se inspeccionarán específicamente dos reportes:

1. `9383196`, debido a la multiplicidad extrema observada para `TOCILIZUMAB`;
2. `15551360`, porque contiene numerosos medicamentos con multiplicidades elevadas.

Para cada elemento `<drug>` se extraerán, cuando estén disponibles:

- `drugcharacterization`;
- `medicinalproduct`;
- `activesubstancename`;
- `drugindication`;
- `drugadministrationroute`;
- `drugdosageform`;
- `drugdosagetext`;
- `drugstructuredosagenumb`;
- `drugstructuredosageunit`;
- `drugseparatedosagenumb`;
- `drugintervaldosageunitnumb`;
- `drugintervaldosagedefinition`;
- `drugtreatmentduration`;
- `drugtreatmentdurationunit`;
- `actiondrug`;
- `drugadditional`;
- `drugauthorizationnumb`.

El objetivo es determinar si las repeticiones corresponden a filas exactamente
iguales o si cada registro conserva información diferente.



### 11.4 Decisión metodológica que se evaluará

Si las filas repetidas contienen información diferente pero corresponden a la misma
sustancia activa, se conservarán en una **tabla detallada de medicamentos**, pero
se deduplicarán en la **tabla analítica de exposición**.

Tendríamos entonces dos niveles de información:

$$
\text{tabla detallada}
\longrightarrow
\text{todos los elementos `<drug>`}
$$

y

$$
\text{tabla analítica}
\longrightarrow
(\texttt{safetyreportid},\texttt{drug\_key}).
$$

Esta separación permitiría conservar toda la información original sin introducir
duplicaciones en los conteos utilizados para PRR y ROR.



### Objetivo de esta sección

Esta auditoría permitirá verificar empíricamente que la deduplicación propuesta no
elimina medicamentos diferentes, sino solamente múltiples registros XML asociados
con una misma sustancia dentro de un mismo reporte.

Después de esta comprobación se podrá establecer formalmente la definición de
`drug_key` que se utilizará en el resto del proyecto.

In [28]:
# 11. Auditoría de reportes con multiplicidad extrema


# Reportes seleccionados para inspección
reportes_auditar = {
    "9383196",
    "15551360"
}


# 11.1 Campos de medicamento que queremos inspeccionar

campos_drug_auditoria = [
    "drugcharacterization",
    "medicinalproduct",
    "activesubstancename",
    "drugindication",
    "drugadministrationroute",
    "drugdosageform",
    "drugdosagetext",
    "drugstructuredosagenumb",
    "drugstructuredosageunit",
    "drugseparatedosagenumb",
    "drugintervaldosageunitnumb",
    "drugintervaldosagedefinition",
    "drugtreatmentduration",
    "drugtreatmentdurationunit",
    "actiondrug",
    "drugadditional",
    "drugauthorizationnumb",
]


# 11.2 Volver a leer incrementalmente el primer XML hasta localizar los reportes seleccionados

registros_auditoria = []
reportes_encontrados = set()

context = ET.iterparse(
    primer_xml,
    events=("end",)
)

for event, elem in context:

    if limpiar_tag(elem.tag).lower() != "safetyreport":
        continue

    safetyreportid = obtener_texto(
        elem,
        "safetyreportid"
    )

    # Solo procesamos los reportes seleccionados
    if safetyreportid not in reportes_auditar:
        elem.clear()
        continue


    # Localizar patient
    patient = None

    for hijo in elem:

        if limpiar_tag(hijo.tag).lower() == "patient":
            patient = hijo
            break


    # Extraer todos los registros <drug>
    if patient is not None:

        drug_n = 0

        for drug in patient:

            if limpiar_tag(drug.tag).lower() != "drug":
                continue

            drug_n += 1

            registro = {
                "safetyreportid": safetyreportid,
                "drug_n": drug_n
            }

            for campo in campos_drug_auditoria:

                registro[campo] = obtener_texto(
                    drug,
                    campo
                )

            registros_auditoria.append(
                registro
            )


    reportes_encontrados.add(
        safetyreportid
    )

    elem.clear()


    # Detener la lectura cuando encontremos ambos reportes
    if reportes_encontrados == reportes_auditar:
        break


# 11.3 Construir DataFrame
df_auditoria_drug = pd.DataFrame(
    registros_auditoria
)


print("REPORTES LOCALIZADOS")
print(sorted(reportes_encontrados))

print()

print(
    f"Registros <drug> extraídos para auditoría: "
    f"{len(df_auditoria_drug):,}"
)

REPORTES LOCALIZADOS
['15551360', '9383196']

Registros <drug> extraídos para auditoría: 473


In [30]:
# 11.4 Resumen de medicamentos por reporte

resumen_auditoria = (
    df_auditoria_drug
    .groupby(
        [
            "safetyreportid",
            "activesubstancename"
        ],
        dropna=False
    )
    .agg(
        n_registros=("drug_n", "count"),
        n_indicaciones=("drugindication", "nunique"),
        n_rutas=("drugadministrationroute", "nunique"),
        n_dosis_texto=("drugdosagetext", "nunique"),
        n_formas=("drugdosageform", "nunique")
    )
    .reset_index()
    .sort_values(
        [
            "safetyreportid",
            "n_registros"
        ],
        ascending=[True, False]
    )
)


print("\nRESUMEN DE MULTIPLICIDAD POR MEDICAMENTO")
display(resumen_auditoria)


# 11.5 Inspección específica de TOCILIZUMAB en safetyreportid = 9383196

tocilizumab_9383196 = (
    df_auditoria_drug[
        (df_auditoria_drug["safetyreportid"] == "9383196")
        &
        (
            df_auditoria_drug[
                "activesubstancename"
            ] == "TOCILIZUMAB"
        )
    ]
    .copy()
)


print(
    "\nTOCILIZUMAB EN EL REPORTE 9383196"
)

print(
    f"Número de registros: "
    f"{len(tocilizumab_9383196):,}"
)

display(
    tocilizumab_9383196
    .head(30)
)


# 11.6 ¿Cuántas filas exactamente iguales existen?

columnas_comparacion = [
    col
    for col in campos_drug_auditoria
    if col != "drugcharacterization"
]

n_filas = len(
    tocilizumab_9383196
)

n_combinaciones_distintas = (
    tocilizumab_9383196[
        columnas_comparacion
    ]
    .drop_duplicates()
    .shape[0]
)


print(
    "\nDIVERSIDAD INTERNA DE LOS REGISTROS "
    "DE TOCILIZUMAB"
)

print(
    f"Filas XML: {n_filas:,}"
)

print(
    f"Combinaciones distintas de atributos: "
    f"{n_combinaciones_distintas:,}"
)


# 11.7 Valores distintos de cada atributo

resumen_campos_tocilizumab = []

for campo in campos_drug_auditoria:

    valores = (
        tocilizumab_9383196[campo]
        .dropna()
        .unique()
        .tolist()
    )

    resumen_campos_tocilizumab.append(
        {
            "campo": campo,
            "n_valores_distintos": len(valores),
            "ejemplos": valores[:10]
        }
    )


df_campos_tocilizumab = pd.DataFrame(
    resumen_campos_tocilizumab
)


print(
    "\nVARIACIÓN DE LOS ATRIBUTOS "
    "DE TOCILIZUMAB"
)

display(df_campos_tocilizumab)


# 11.8 Medicamentos del reporte 15551360

reporte_15551360 = (
    resumen_auditoria[
        resumen_auditoria[
            "safetyreportid"
        ] == "15551360"
    ]
    .reset_index(drop=True)
)


print(
    "\nMEDICAMENTOS DEL REPORTE 15551360"
)

display(reporte_15551360)


RESUMEN DE MULTIPLICIDAD POR MEDICAMENTO


,safetyreportid,activesubstancename,n_registros,n_indicaciones,n_rutas,n_dosis_texto,n_formas
16,15551360,LOXAPINE SUCCINATE,32,3,2,5,2
17,15551360,OLANZAPINE,26,2,1,4,1
21,15551360,RISPERIDONE,26,2,1,3,1
18,15551360,PERPHENAZINE,24,3,1,2,2
14,15551360,LITHIUM CARBONATE,23,2,1,2,1
24,15551360,VALPROATE SODIUM,22,3,2,0,2
10,15551360,GABAPENTIN,20,2,1,1,2
1,15551360,ARIPIPRAZOLE,18,3,2,1,2
23,15551360,TRIFLUOPERAZINE HYDROCHLORIDE,18,3,1,1,2
20,15551360,QUETIAPINE FUMARATE,16,2,3,1,3



TOCILIZUMAB EN EL REPORTE 9383196
Número de registros: 85


,safetyreportid,drug_n,drugcharacterization,medicinalproduct,activesubstancename,drugindication,drugadministrationroute,drugdosageform,drugdosagetext,drugstructuredosagenumb,drugstructuredosageunit,drugseparatedosagenumb,drugintervaldosageunitnumb,drugintervaldosagedefinition,drugtreatmentduration,drugtreatmentdurationunit,actiondrug,drugadditional,drugauthorizationnumb
378,9383196,1,1,ACTEMRA,TOCILIZUMAB,Rheumatoid arthritis,042,Solution for infusion,DATE OF LAST DOSE: 02/MAY/2012?FIRST RPAP INFU...,424,003,1,4,803,85,804,5,3,125276
379,9383196,2,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,208,003,1,4,803,85,804,5,3,125276
380,9383196,3,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,DATE OF MOST RECENT DOSE: 26/JUN/2013.,260,003,1,4,803,174,804,5,3,125276
381,9383196,4,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,208,003,1,4,803,50,804,5,3,125276
382,9383196,5,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,260,003,1,4,803,NaN,NaN,5,3,125276
383,9383196,6,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,208,003,1,4,803,498,804,5,3,125276
384,9383196,7,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,324,003,1,4,803,5,801,5,3,125276
385,9383196,8,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,319,003,1,4,803,246,804,5,3,125276
386,9383196,9,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,3,125276
387,9383196,10,1,ACTEMRA,TOCILIZUMAB,NaN,042,Solution for infusion,NaN,324,003,1,4,803,NaN,NaN,5,3,125276



DIVERSIDAD INTERNA DE LOS REGISTROS DE TOCILIZUMAB
Filas XML: 85
Combinaciones distintas de atributos: 28

VARIACIÓN DE LOS ATRIBUTOS DE TOCILIZUMAB


,campo,n_valores_distintos,ejemplos
0,drugcharacterization,1,[1]
1,medicinalproduct,1,[ACTEMRA]
2,activesubstancename,1,[TOCILIZUMAB]
3,drugindication,1,[Rheumatoid arthritis]
4,drugadministrationroute,1,[042]
5,drugdosageform,1,[Solution for infusion]
6,drugdosagetext,2,[DATE OF LAST DOSE: 02/MAY/2012?FIRST RPAP INF...
7,drugstructuredosagenumb,17,"[424, 208, 260, 324, 319, 321, 320.4, 319.2, 3..."
8,drugstructuredosageunit,1,[003]
9,drugseparatedosagenumb,1,[1]



MEDICAMENTOS DEL REPORTE 15551360


,safetyreportid,activesubstancename,n_registros,n_indicaciones,n_rutas,n_dosis_texto,n_formas
0,15551360,LOXAPINE SUCCINATE,32,3,2,5,2
1,15551360,OLANZAPINE,26,2,1,4,1
2,15551360,RISPERIDONE,26,2,1,3,1
3,15551360,PERPHENAZINE,24,3,1,2,2
4,15551360,LITHIUM CARBONATE,23,2,1,2,1
5,15551360,VALPROATE SODIUM,22,3,2,0,2
6,15551360,GABAPENTIN,20,2,1,1,2
7,15551360,ARIPIPRAZOLE,18,3,2,1,2
8,15551360,TRIFLUOPERAZINE HYDROCHLORIDE,18,3,1,1,2
9,15551360,QUETIAPINE FUMARATE,16,2,3,1,3


La auditoría resolvió la duda principal. En el reporte `9383196`, `TOCILIZUMAB` aparece 85 veces, pero esas 85 filas corresponden a 28 combinaciones distintas de atributos. El medicamento, principio activo, indicación, vía y forma farmacéutica permanecen esencialmente constantes; lo que cambia sobre todo es la dosis y la duración del tratamiento: encontramos 17 valores distintos de dosis estructurada y 11 duraciones diferentes. 

Por tanto, no son 85 exposiciones independientes a TOCILIZUMAB. Son múltiples registros clínicos/terapéuticos del mismo medicamento dentro del mismo reporte. Para PRR y ROR, ese caso debe aportar una sola presencia:

$$
X_{iD}=1.
$$

El reporte `15551360` confirma el mismo fenómeno a mayor escala: contiene numerosos medicamentos repetidos, asociados con distintas indicaciones, rutas, dosis y formas farmacéuticas. 

Hay además un detalle importante: en la tabla de auditoría algunos conteos son ligeramente mayores que en nuestra tabla anterior, por ejemplo `LOXAPINE SUCCINATE` aparece 32 veces aquí frente a 31 en la cohorte `Suspect`, porque la auditoría recuperó todos los elementos `<drug>`, mientras que la tabla anterior estaba filtrada a `drugcharacterization = 1`. No es una inconsistencia; de hecho, confirma que debemos aplicar el filtro `Suspect` antes de construir la tabla analítica.

Con esto ya podemos fijar nuestra estructura:

* **Tabla detallada:** conserva todos los `<drug>` y su información.
* **Tabla analítica de exposición:** una sola fila por $(\text{reporte},\text{medicamento})$.
* Cohorte principal: `drugcharacterization = 1`.
* `drug_key`: preferentemente `activesubstancename`; `medicinalproduct` queda como respaldo por robustez para cuando procesemos todos los XML.
* Normalización: únicamente mayúsculas/espacios por ahora.
* Combinaciones como `BUDESONIDE\FORMOTEROL\GLYCOPYRROLATE` se mantienen como una entidad, sin separarlas todavía.

## 12. Definición de la unidad analítica medicamento--reporte

La auditoría de casos con multiplicidad elevada confirmó que un mismo medicamento
puede aparecer numerosas veces dentro de un único reporte debido a diferencias en
información asociada con la exposición.

Por ejemplo, en uno de los casos auditados se encontraron 85 elementos `<drug>` correspondientes a `TOCILIZUMAB`.

Sin embargo, estos registros compartían el mismo medicamento y la misma sustancia
activa, mientras que las diferencias se concentraban principalmente en variables
como dosis y duración del tratamiento.

Por tanto, los elementos `<drug>` representan **registros de información sobre el
medicamento**, pero no necesariamente exposiciones independientes.

Esta distinción es fundamental para el análisis de desproporcionalidad.



### 12.1 Tabla detallada y tabla analítica

A partir de este punto se distinguirán dos niveles de datos.

#### Tabla detallada de medicamentos

La tabla detallada conservará todos los elementos `<drug>` encontrados en los
archivos XML.

Conceptualmente tendrá la estructura

$$
\texttt{df\_suspect\_detalle}
=
\{
\text{todos los registros `<drug>` clasificados como Suspect}
\}.
$$

Esta tabla conservará variables como:

- producto medicinal;
- sustancia activa;
- indicación;
- dosis;
- vía de administración;
- otras características del medicamento.

Su objetivo será mantener la trazabilidad de la información original.

#### Tabla analítica de exposición

Para el cálculo de señales se construirá una segunda tabla donde cada combinación

$$
(\texttt{safetyreportid},\texttt{drug\_key})
$$

aparezca una sola vez.

La presencia de un medicamento $D$ en un reporte $i$ se representará mediante

$$
X_{iD}
=
\begin{cases}
1, & \text{si }D\text{ aparece como Suspect en el reporte }i,\\
0, & \text{en otro caso}.
\end{cases}
$$

Por tanto, aunque un medicamento aparezca varias veces dentro del XML,

$$
r_{iD}>1,
$$

su contribución al análisis seguirá siendo

$$
X_{iD}=1.
$$

Esta definición evita que diferencias en dosis, indicación, duración o vía de
administración aumenten artificialmente los conteos utilizados para calcular
medidas de desproporcionalidad.


### 12.2 Definición de `drug_key`

El identificador analítico del medicamento se denominará

`drug_key`.

Se construirá mediante la regla jerárquica

$$
\texttt{drug\_key}
=
\begin{cases}
\texttt{activesubstancename},
&
\text{si está disponible},
\\[6pt]
\texttt{medicinalproduct},
&
\text{en otro caso}.
\end{cases}
$$

Posteriormente se aplicará únicamente una normalización textual conservadora:

1. eliminación de espacios al inicio y al final;
2. conversión a mayúsculas;
3. sustitución de espacios consecutivos por un único espacio.

No se eliminarán por ahora:

- sales farmacéuticas;
- sufijos químicos;
- signos de puntuación;
- combinaciones de sustancias.

Por ejemplo,

`PIMAVANSERIN TARTRATE`

y

`PIMAVANSERIN`

permanecerán como identificadores diferentes.

De igual manera, una combinación como

`BUDESONIDE\FORMOTEROL\GLYCOPYRROLATE`

se conservará inicialmente como una sola entidad farmacológica.



### 12.3 Multiplicidad como información auxiliar

Aunque la tabla analítica tendrá una sola fila por reporte--medicamento,
conservaremos una variable

`n_xml_rows`

definida como

$$
r_{iD}
=
\text{número de elementos `<drug>` del medicamento }D
\text{ encontrados en el reporte }i.
$$

Esta variable no será utilizada como peso en PRR o ROR.

Su función será exclusivamente diagnóstica y permitirá mantener información sobre
la estructura original del reporte.

También conservaremos:

- número de productos medicinales asociados;
- número de indicaciones distintas.



### 12.4 Regla de deduplicación para el análisis

La unidad fundamental para los medicamentos será entonces

$$
(\texttt{safetyreportid},\texttt{drug\_key}).
$$

Posteriormente, cuando se incorporen los eventos adversos, la unidad de análisis
se extenderá a

$$
(
\texttt{safetyreportid},
\texttt{drug\_key},
\texttt{reaction\_pt}
).
$$

Cada combinación reporte--medicamento--evento deberá aparecer como máximo una vez
antes de calcular los conteos de desproporcionalidad.

Esta decisión permite evitar pseudorreplicación sin eliminar la información
detallada original.



### Objetivo de esta sección

En esta etapa se construirá formalmente, para la muestra de $1{,}000$ reportes:

1. la tabla detallada de medicamentos sospechosos;
2. la variable definitiva `drug_key`;
3. la tabla analítica de exposiciones únicas;
4. la variable auxiliar `n_xml_rows`;
5. verificaciones automáticas para asegurar que no existan pares
   reporte--medicamento duplicados.

Esta misma lógica será utilizada posteriormente para procesar el conjunto completo
de archivos FAERS.

In [32]:
# 12. Construcción formal de la tabla analítica reporte-medicamento


# 12.1 Tabla detallada de medicamentos Suspect

df_suspect_detalle = (
    df_muestra_medicamentos[
        df_muestra_medicamentos[
            "drugcharacterization"
        ] == "1"
    ]
    .copy()
)


# 12.2 Construcción de drug_key

# Regla jerárquica:
# 1. activesubstancename
# 2. medicinalproduct como respaldo

df_suspect_detalle["drug_key"] = (
    df_suspect_detalle[
        "activesubstancename"
    ]
    .fillna(
        df_suspect_detalle[
            "medicinalproduct"
        ]
    )
    .apply(
        normalizar_drug_text_minimo
    )
)


# 12.3 Verificar disponibilidad de drug_key

n_suspect = len(df_suspect_detalle)

n_key_faltante = (
    df_suspect_detalle[
        "drug_key"
    ]
    .isna()
    .sum()
)


print("TABLA DETALLADA SUSPECT")
print(
    f"Registros <drug> Suspect: "
    f"{n_suspect:,}"
)

print(
    f"Registros sin drug_key: "
    f"{n_key_faltante:,}"
)


# 12.4 Construir tabla analítica de exposición

df_suspect_exposicion = (
    df_suspect_detalle
    .dropna(
        subset=["drug_key"]
    )
    .groupby(
        [
            "safetyreportid",
            "drug_key"
        ],
        as_index=False
    )
    .agg(
        # Número de filas XML originales
        n_xml_rows=(
            "drug_n",
            "count"
        ),

        # Número de nombres de producto distintos
        n_medicinalproducts=(
            "medicinalproduct",
            "nunique"
        ),

        # Número de indicaciones distintas
        n_indicaciones=(
            "drugindication",
            "nunique"
        )
    )
)


# 12.5 Verificaciones de integridad

n_exposiciones = len(
    df_suspect_exposicion
)

n_reportes_suspect = (
    df_suspect_exposicion[
        "safetyreportid"
    ]
    .nunique()
)

n_drugs_unicos = (
    df_suspect_exposicion[
        "drug_key"
    ]
    .nunique()
)


duplicados_finales = (
    df_suspect_exposicion
    .duplicated(
        subset=[
            "safetyreportid",
            "drug_key"
        ]
    )
    .sum()
)


print(
    "\nTABLA ANALÍTICA DE EXPOSICIÓN"
)

print(
    f"Pares únicos reporte-medicamento: "
    f"{n_exposiciones:,}"
)

print(
    f"Reportes representados: "
    f"{n_reportes_suspect:,}"
)

print(
    f"Medicamentos distintos: "
    f"{n_drugs_unicos:,}"
)

print(
    f"Pares duplicados después de agrupar: "
    f"{duplicados_finales:,}"
)


# 12.6 Comprobar que se conservaron todas las filas XML mediante n_xml_rows

filas_reconstruidas = (
    df_suspect_exposicion[
        "n_xml_rows"
    ]
    .sum()
)


print(
    "\nVERIFICACIÓN DE TRAZABILIDAD"
)

print(
    f"Filas Suspect originales: "
    f"{len(df_suspect_detalle):,}"
)

print(
    f"Suma de n_xml_rows: "
    f"{filas_reconstruidas:,}"
)

print(
    "¿Coinciden?:",
    filas_reconstruidas
    == len(df_suspect_detalle)
)


# 12.7 Distribución de n_xml_rows

print(
    "\nDISTRIBUCIÓN DE n_xml_rows"
)

display(
    df_suspect_exposicion[
        "n_xml_rows"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 12.8 Exposiciones con mayor número de filas XML

print(
    "\nEXPOSICIONES CON MAYOR MULTIPLICIDAD XML"
)

display(
    df_suspect_exposicion
    .sort_values(
        "n_xml_rows",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

TABLA DETALLADA SUSPECT
Registros <drug> Suspect: 2,938
Registros sin drug_key: 0

TABLA ANALÍTICA DE EXPOSICIÓN
Pares únicos reporte-medicamento: 1,589
Reportes representados: 995
Medicamentos distintos: 538
Pares duplicados después de agrupar: 0

VERIFICACIÓN DE TRAZABILIDAD
Filas Suspect originales: 2,938
Suma de n_xml_rows: 2,938
¿Coinciden?: True

DISTRIBUCIÓN DE n_xml_rows


,valor
count,1589.000000
mean,1.848962
std,3.043115
min,1.000000
50%,1.000000
75%,2.000000
90%,3.000000
95%,4.000000
99%,12.120000
max,85.000000



EXPOSICIONES CON MAYOR MULTIPLICIDAD XML


,safetyreportid,drug_key,n_xml_rows,n_medicinalproducts,n_indicaciones
0,9383196,TOCILIZUMAB,85,1,1
1,15551360,LOXAPINE SUCCINATE,31,1,2
2,15551360,RISPERIDONE,26,1,2
3,15551360,OLANZAPINE,26,1,2
4,15551360,PERPHENAZINE,23,1,2
5,15551360,LITHIUM CARBONATE,23,1,2
6,15688545,QUETIAPINE,19,1,1
7,15551360,GABAPENTIN,19,1,1
8,15551360,ARIPIPRAZOLE,17,2,2
9,15551360,TRIFLUOPERAZINE HYDROCHLORIDE,17,2,2


Los resultados cerraron la parte de identificación de medicamentos.

La tabla analítica quedó exactamente como queríamos: los $2{,}938$ registros `<drug>` sospechosos se reducen a $1{,}589$ exposiciones únicas reporte–medicamento, sin perder trazabilidad porque

$$
\sum n_{\text{xml_rows}}=2938.
$$

Además, después de agrupar tenemos **0 pares duplicados**, por lo que la unidad

$$
(\texttt{safetyreportid},\texttt{drug_key})
$$

queda correctamente definida. También es interesante que algunos principios activos, como `ARIPIPRAZOLE`, aparecen asociados con más de un `medicinalproduct`; eso refuerza la decisión de utilizar la sustancia activa como identificador principal.

## 13. Estudio de las fechas `receivedate` y `receiptdate`

Una vez definida la unidad analítica medicamento--reporte, el siguiente aspecto
fundamental del proyecto es establecer correctamente la dimensión temporal.

Hasta ahora sabemos que los datos analizados provienen del archivo

`1_ADR25Q1.xml`,

el cual pertenece al extracto trimestral $2025Q1.$

Sin embargo, en el primer reporte inspeccionado se observó

$$
\texttt{receivedate}=20241210
$$

y

$$
\texttt{receiptdate}=20241210.
$$

Es decir, la fecha registrada dentro del reporte pertenece a diciembre de 2024,
aunque el caso aparece dentro del extracto correspondiente al primer trimestre de
2025.

Esta situación no debe considerarse automáticamente un error. FAERS utiliza una
estructura basada en casos y versiones, por lo que el trimestre en el que un caso
aparece dentro del *Quarterly Data Extract* no necesariamente coincide de manera
exacta con la fecha registrada dentro del reporte.

Por esta razón, antes de construir las series temporales del proyecto es necesario
comprender qué información contienen ambas fechas.



### 13.1 Dos conceptos temporales diferentes

En los archivos XML disponemos, entre otras, de las variables:

- `receivedate`;
- `receiptdate`.

Estas variables no deben suponerse equivalentes sin comprobarlo empíricamente.

Para cada reporte $i$ podemos representar ambas fechas como

$$
R_i^{(\mathrm{initial})}
=
\texttt{receivedate}_i
$$

y

$$
R_i^{(\mathrm{receipt})}
=
\texttt{receiptdate}_i.
$$

Además, existe una tercera referencia temporal que no procede directamente de una
etiqueta del reporte:

$$
Q_i^{(\mathrm{QDE})},
$$

que corresponde al trimestre del archivo en el que encontramos el reporte.

En la muestra actual,

$$
Q_i^{(\mathrm{QDE})}=2025Q1
$$

para los $1{,}000$ reportes.

Por tanto, el proyecto dispone potencialmente de tres nociones temporales:

$$
\text{fecha inicial},
\qquad
\text{fecha de recepción},
\qquad
\text{trimestre del extracto}.
$$

Antes de seleccionar una de ellas para el análisis longitudinal debemos estudiar sus
relaciones.


### 13.2 Formato de las fechas

Las fechas FAERS suelen almacenarse como cadenas de caracteres con formato

$$
YYYYMMDD.
$$

Por ejemplo,

$$
20241210
\longrightarrow
10\text{ de diciembre de }2024.
$$

Sin embargo, en algunas variables FAERS pueden existir fechas parciales.

Por ello, antes de convertir automáticamente las cadenas a objetos de fecha,
analizaremos:

1. cuántos valores están ausentes;
2. qué longitudes presentan las cadenas;
3. cuántos tienen $4$, $6$ u $8$ caracteres;
4. si existen formatos inesperados.

Solamente después de esta validación realizaremos la conversión.


### 13.3 Diferencias entre ambas fechas

Para los reportes donde ambas fechas estén disponibles calcularemos

$$
\Delta_i
=
\texttt{receiptdate}_i
-
\texttt{receivedate}_i.
$$

La diferencia $\Delta_i$ se expresará en días.

Si

$$
\Delta_i=0,
$$

ambas fechas coinciden.

Si

$$
\Delta_i>0,
$$

`receiptdate` es posterior a `receivedate`.

El objetivo no será interpretar todavía clínicamente esta diferencia, sino conocer
su distribución dentro de los datos.



### 13.4 Relación con el trimestre QDE

También clasificaremos las fechas según el trimestre calendario al que pertenecen.

Por ejemplo,

$$
2024\text{-}12\text{-}10
\longrightarrow
2024Q4.
$$

Después podremos comparar:

$$
Q_i^{(\mathrm{fecha})}
\quad\text{vs.}\quad
Q_i^{(\mathrm{QDE})}.
$$

En esta muestra, el segundo valor es siempre

$$
2025Q1.
$$

Queremos conocer qué proporción de reportes presenta:

$$
Q_i^{(\mathrm{fecha})}=2025Q1
$$

y qué proporción corresponde a periodos anteriores.



### 13.5 Objetivo de esta sección

Esta etapa permitirá responder:

1. ¿están completas `receivedate` y `receiptdate`?;
2. ¿qué formatos presentan?;
3. ¿con qué frecuencia ambas fechas coinciden?;
4. ¿qué tan grandes son sus diferencias cuando no coinciden?;
5. ¿cuál es la fecha mínima y máxima observada?;
6. ¿qué proporción de los reportes del extracto `2025Q1` tiene fechas anteriores a
   2025?;
7. ¿a qué trimestres calendario pertenecen realmente las fechas de los reportes?

No se elegirá todavía la variable temporal definitiva.

Primero se caracterizarán las fechas y, a partir de los resultados, se establecerá
la estrategia temporal que se utilizará en el análisis de señales.

In [34]:
# 13. Estudio de receivedate y receiptdate


# 13.1 Copia de trabajo

df_fechas = (
    df_muestra_reportes[
        [
            "safetyreportid",
            "safetyreportversion",
            "receivedate",
            "receiptdate",
            "occurcountry"
        ]
    ]
    .copy()
)

# Trimestre del archivo que estamos estudiando
df_fechas["qde_period"] = "2025Q1"


# 13.2 Inspección de formatos originales

for columna in ["receivedate", "receiptdate"]:

    df_fechas[f"{columna}_length"] = (
        df_fechas[columna]
        .astype("string")
        .str.len()
    )


print("VALORES FALTANTES")

for columna in ["receivedate", "receiptdate"]:

    n_missing = df_fechas[columna].isna().sum()

    print(
        f"{columna}: "
        f"{n_missing:,} "
        f"({100*n_missing/len(df_fechas):.2f}%)"
    )


# 13.3 Distribución de longitudes

print("\nLONGITUDES DE receivedate")

display(
    df_fechas[
        "receivedate_length"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)


print("\nLONGITUDES DE receiptdate")

display(
    df_fechas[
        "receiptdate_length"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)


# 13.4 Función de conversión conservadora

def convertir_fecha_faers(valor):
    """
    Convierte fechas FAERS únicamente cuando tienen
    formato completo YYYYMMDD.

    Las fechas parciales se dejan como NaT para evitar
    inventar día o mes.
    """

    if pd.isna(valor):
        return pd.NaT

    valor = str(valor).strip()

    if len(valor) != 8:
        return pd.NaT

    return pd.to_datetime(
        valor,
        format="%Y%m%d",
        errors="coerce"
    )


df_fechas["receivedate_dt"] = (
    df_fechas["receivedate"]
    .apply(convertir_fecha_faers)
)

df_fechas["receiptdate_dt"] = (
    df_fechas["receiptdate"]
    .apply(convertir_fecha_faers)
)


# 13.5 Conversión exitosa

print("\nCONVERSIÓN A FECHA COMPLETA")

for columna in [
    "receivedate_dt",
    "receiptdate_dt"
]:

    n_valid = df_fechas[columna].notna().sum()

    print(
        f"{columna}: "
        f"{n_valid:,} fechas válidas "
        f"({100*n_valid/len(df_fechas):.2f}%)"
    )


# 13.6 Intervalos temporales observados

print("\nRANGO TEMPORAL")

print(
    "receivedate:",
    df_fechas["receivedate_dt"].min(),
    "->",
    df_fechas["receivedate_dt"].max()
)

print(
    "receiptdate:",
    df_fechas["receiptdate_dt"].min(),
    "->",
    df_fechas["receiptdate_dt"].max()
)


# 13.7 Comparación entre ambas fechas

mask_ambas = (
    df_fechas["receivedate_dt"].notna()
    &
    df_fechas["receiptdate_dt"].notna()
)

df_fechas.loc[
    mask_ambas,
    "delta_dias"
] = (
    df_fechas.loc[
        mask_ambas,
        "receiptdate_dt"
    ]
    -
    df_fechas.loc[
        mask_ambas,
        "receivedate_dt"
    ]
).dt.days


n_ambas = mask_ambas.sum()

n_iguales = (
    df_fechas.loc[
        mask_ambas,
        "delta_dias"
    ] == 0
).sum()

n_diferentes = n_ambas - n_iguales


print("\nCOMPARACIÓN receivedate vs receiptdate")

print(
    f"Reportes con ambas fechas completas: "
    f"{n_ambas:,}"
)

print(
    f"Fechas idénticas: "
    f"{n_iguales:,} "
    f"({100*n_iguales/n_ambas:.2f}%)"
)

print(
    f"Fechas diferentes: "
    f"{n_diferentes:,} "
    f"({100*n_diferentes/n_ambas:.2f}%)"
)


# 13.8 Distribución de la diferencia en días

print("\nDISTRIBUCIÓN DE delta_dias")

display(
    df_fechas.loc[
        mask_ambas,
        "delta_dias"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 13.9 Crear trimestre calendario de cada fecha

df_fechas["receivedate_quarter"] = (
    df_fechas["receivedate_dt"]
    .dt.to_period("Q")
    .astype("string")
)

df_fechas["receiptdate_quarter"] = (
    df_fechas["receiptdate_dt"]
    .dt.to_period("Q")
    .astype("string")
)


# 13.10 Distribución trimestral de receivedate

tabla_receivedate_q = (
    df_fechas[
        "receivedate_quarter"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("trimestre")
    .reset_index(name="n_reportes")
)

tabla_receivedate_q["porcentaje"] = (
    100
    * tabla_receivedate_q["n_reportes"]
    / len(df_fechas)
)


print("\nTRIMESTRE CALENDARIO DE receivedate")

display(tabla_receivedate_q)


# 13.11 Distribución trimestral de receiptdate

tabla_receiptdate_q = (
    df_fechas[
        "receiptdate_quarter"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("trimestre")
    .reset_index(name="n_reportes")
)

tabla_receiptdate_q["porcentaje"] = (
    100
    * tabla_receiptdate_q["n_reportes"]
    / len(df_fechas)
)


print("\nTRIMESTRE CALENDARIO DE receiptdate")

display(tabla_receiptdate_q)


# 13.12 Reportes anteriores a 2025

inicio_2025 = pd.Timestamp("2025-01-01")

n_receivedate_pre2025 = (
    df_fechas["receivedate_dt"]
    < inicio_2025
).sum()

n_receiptdate_pre2025 = (
    df_fechas["receiptdate_dt"]
    < inicio_2025
).sum()


print("\nREPORTES CON FECHAS ANTERIORES A 2025")

print(
    f"receivedate anterior a 2025: "
    f"{n_receivedate_pre2025:,} "
    f"({100*n_receivedate_pre2025/len(df_fechas):.2f}%)"
)

print(
    f"receiptdate anterior a 2025: "
    f"{n_receiptdate_pre2025:,} "
    f"({100*n_receiptdate_pre2025/len(df_fechas):.2f}%)"
)


# 13.13 Ejemplos con mayor diferencia entre fechas

print(
    "\nREPORTES CON MAYOR DIFERENCIA ENTRE "
    "receivedate Y receiptdate"
)

display(
    df_fechas[
        [
            "safetyreportid",
            "safetyreportversion",
            "receivedate",
            "receiptdate",
            "delta_dias",
            "receivedate_quarter",
            "receiptdate_quarter",
            "qde_period"
        ]
    ]
    .sort_values(
        "delta_dias",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

VALORES FALTANTES
receivedate: 0 (0.00%)
receiptdate: 0 (0.00%)

LONGITUDES DE receivedate


,n_caracteres,n_reportes
0,8,1000



LONGITUDES DE receiptdate


,n_caracteres,n_reportes
0,8,1000



CONVERSIÓN A FECHA COMPLETA
receivedate_dt: 1,000 fechas válidas (100.00%)
receiptdate_dt: 1,000 fechas válidas (100.00%)

RANGO TEMPORAL
receivedate: 2013-07-04 00:00:00 -> 2025-01-01 00:00:00
receiptdate: 2024-12-10 00:00:00 -> 2025-01-01 00:00:00

COMPARACIÓN receivedate vs receiptdate
Reportes con ambas fechas completas: 1,000
Fechas idénticas: 621 (62.10%)
Fechas diferentes: 379 (37.90%)

DISTRIBUCIÓN DE delta_dias


...

,trimestre,n_reportes,porcentaje
0,2013Q3,1,0.1
1,2017Q2,1,0.1
2,2018Q4,2,0.2
3,2019Q1,1,0.1
4,2019Q4,1,0.1
5,2020Q3,5,0.5
6,2020Q4,3,0.3
7,2021Q2,3,0.3
8,2021Q3,3,0.3
9,2021Q4,3,0.3



TRIMESTRE CALENDARIO DE receiptdate


,trimestre,n_reportes,porcentaje
0,2024Q4,2,0.2
1,2025Q1,998,99.8



REPORTES CON FECHAS ANTERIORES A 2025
receivedate anterior a 2025: 381 (38.10%)
receiptdate anterior a 2025: 2 (0.20%)

REPORTES CON MAYOR DIFERENCIA ENTRE receivedate Y receiptdate


,safetyreportid,safetyreportversion,receivedate,receiptdate,delta_dias,receivedate_quarter,receiptdate_quarter,qde_period
0,9383196,57,20130704,20250101,4199.0,2013Q3,2025Q1,2025Q1
1,13599008,10,20170531,20250101,2772.0,2017Q2,2025Q1,2025Q1
2,15551360,17,20181025,20250101,2260.0,2018Q4,2025Q1,2025Q1
3,15688545,18,20181205,20250101,2219.0,2018Q4,2025Q1,2025Q1
4,15841513,16,20190118,20250101,2175.0,2019Q1,2025Q1,2025Q1
5,17196061,10,20191224,20250101,1835.0,2019Q4,2025Q1,2025Q1
6,18130441,15,20200810,20250101,1605.0,2020Q3,2025Q1,2025Q1
7,18136443,14,20200811,20250101,1604.0,2020Q3,2025Q1,2025Q1
8,18219065,31,20200901,20250101,1583.0,2020Q3,2025Q1,2025Q1
9,18273891,2,20200916,20250101,1568.0,2020Q3,2025Q1,2025Q1


Estos resultados aclaran muy bien la función de las dos fechas, pero también muestran algo que debemos corregir metodológicamente antes de sacar una conclusión definitiva.

En los primeros $1{,}000$ reportes, `receivedate` puede remontarse hasta 2013, mientras que `receiptdate` está casi completamente dentro de `2025Q1`. Además, los casos con `receivedate` muy antiguo presentan versiones altas, por ejemplo, el reporte `9383196` tiene versión 57, `receivedate = 2013-07-04` y `receiptdate = 2025-01-01`. Esto es exactamente compatible con la estructura Case/Version documentada por FAERS: `receivedate` permite localizar cuándo se recibió inicialmente el caso, mientras que `receiptdate` corresponde a la recepción de la versión individual del caso. 

Por tanto, la interpretación provisional sería:

$$
\boxed{\texttt{receivedate}\approx\text{fecha inicial/histórica del caso}}
$$

y

$$
\boxed{\texttt{receiptdate}\approx\text{fecha de recepción de la versión incluida}}
$$

mientras que

$$
\boxed{\texttt{qde_period}=\text{trimestre del archivo de procedencia}}.
$$

Esto explica por qué $38.1%$ de los `receivedate` son anteriores a 2025, mientras que solo $0.2%$ de los `receiptdate` lo son.

Sin embargo, **todavía no debemos decidir definitivamente que `receiptdate` será nuestra variable temporal**, porque hay un sesgo en la muestra que acabamos de analizar: tomamos **los primeros 1,000 reportes del primer XML**. El hecho de que la fecha máxima de `receiptdate` sea exactamente `2025-01-01` sugiere fuertemente que los registros están ordenados o parcialmente ordenados por fecha. Por tanto, esos $1{,}000$ casos no representan aleatoriamente todo `2025Q1`.

## 14. Validación de la estructura temporal utilizando el trimestre completo 2025Q1

La exploración de los primeros $1{,}000$ reportes mostró una diferencia importante
entre `receivedate` y `receiptdate`.

En la muestra se observó que:

- `receivedate` puede corresponder a varios años anteriores al extracto;
- `receiptdate` coincide casi siempre con el trimestre del archivo;
- algunos reportes con `receivedate` antiguo presentan números elevados de versión.

Por ejemplo, se identificaron casos cuya primera fecha registrada se remonta varios
años atrás, pero cuya `receiptdate` pertenece a 2025.

Esto es consistente con la estructura **Case/Version** de FAERS: un mismo caso puede
ser actualizado mediante nuevas versiones y la versión más reciente puede ser
recibida mucho tiempo después de la recepción inicial del caso.


### 14.1 Limitación de la muestra anterior

Los resultados anteriores se obtuvieron a partir de los primeros

$$
N=1000
$$

reportes de `1_ADR25Q1.xml`.

Esta muestra permitió estudiar correctamente la estructura de los datos, pero no
debe utilizarse todavía para caracterizar la distribución temporal de todo el
trimestre.

Una evidencia de ello es que la fecha máxima observada de `receiptdate` fue

$$
2025\text{-}01\text{-}01,
$$

aunque el archivo pertenece al trimestre

$$
2025Q1.
$$

Esto sugiere que el orden de los reportes dentro del XML puede estar relacionado con
la fecha de recepción.

En consecuencia, utilizar solamente los primeros registros podría producir una
muestra temporalmente sesgada.



### 14.2 Validación con el trimestre completo

Para evitar este problema se procesarán los tres fragmentos XML correspondientes a

$$
2025Q1:
$$

    1_ADR25Q1.xml
    2_ADR25Q1.xml
    3_ADR25Q1.xml

En esta etapa no extraeremos medicamentos ni reacciones.

De cada elemento `<safetyreport>` únicamente conservaremos:

- `safetyreportid`;
- `safetyreportversion`;
- `receivedate`;
- `receiptdate`;
- `occurcountry`;
- archivo de procedencia.

Esto permitirá recorrer el trimestre completo utilizando una cantidad de memoria
mucho menor que la necesaria para almacenar toda la información farmacológica.


### 14.3 Preguntas que queremos responder

Con todos los reportes de 2025Q1 estudiaremos:

1. ¿Cuántos reportes contienen los tres archivos XML?
2. ¿Cuál es el rango completo de `receiptdate`?
3. ¿Qué proporción de los reportes tiene `receiptdate` dentro de 2025Q1?
4. ¿Cuántos casos presentan `receiptdate` fuera del trimestre del extracto?
5. ¿Qué tan frecuente es que `receivedate` corresponda a años anteriores?
6. ¿Existen `safetyreportid` repetidos entre los tres fragmentos XML?
7. ¿Cuál es la distribución de la diferencia entre ambas fechas?


### 14.4 Tres referencias temporales

A partir de este punto distinguiremos explícitamente tres variables:

$$
Q_i^{(\mathrm{QDE})}
=
\text{trimestre del archivo de procedencia},
$$

$$
Q_i^{(\mathrm{receipt})}
=
\text{trimestre calendario de `receiptdate`},
$$

y

$$
Q_i^{(\mathrm{received})}
=
\text{trimestre calendario de `receivedate`}.
$$

Una posible estrategia para el análisis longitudinal sería utilizar

$$
Q_i^{(\mathrm{receipt})}
$$

como dimensión temporal principal, conservar

$$
Q_i^{(\mathrm{QDE})}
$$

como variable de procedencia y control de calidad, y utilizar `receivedate` como
información sobre la antigüedad o historia del caso.

Sin embargo, esta decisión **todavía es provisional**.

Primero verificaremos empíricamente la relación entre estas tres referencias
temporales utilizando el trimestre completo.


### 14.5 Lectura incremental

Los tres archivos de 2025Q1 ocupan conjuntamente más de $2$ GB.

Por esta razón utilizaremos nuevamente lectura incremental.

Para cada reporte:

1. se leerán únicamente las variables administrativas necesarias;
2. se almacenará una fila de metadatos;
3. se liberará inmediatamente el elemento XML de memoria.

No se construirán todavía las tablas de medicamentos, reacciones ni pares
fármaco--evento.

Esta etapa constituye una validación de la estructura temporal antes de iniciar el
procesamiento completo de los seis trimestres.

In [36]:
# 14. Validación temporal del trimestre completo 2025Q1


# 14.1 Seleccionar los tres XML de 2025Q1
archivos_2025q1 = (
    df_xml[
        df_xml["periodo"] == "2025Q1"
    ]
    .sort_values("parte")
    .copy()
)


print("ARCHIVOS QUE SERÁN PROCESADOS")

display(
    archivos_2025q1[
        [
            "parte",
            "archivo",
            "size_mb"
        ]
    ]
)


# 14.2 Contenedor para los metadatos

registros_metadata_q1 = []

conteos_por_archivo = []


# 14.3 Procesamiento incremental

for _, fila_archivo in archivos_2025q1.iterrows():

    ruta_xml = Path(
        fila_archivo["ruta"]
    )

    nombre_archivo = fila_archivo["archivo"]
    parte = int(fila_archivo["parte"])

    print()
    print("=" * 70)
    print(f"Procesando: {nombre_archivo}")
    print("=" * 70)

    contador = 0
    root = None

    context = ET.iterparse(
        ruta_xml,
        events=("start", "end")
    )

    for event, elem in context:

        # Guardar referencia al elemento raíz

        if root is None and event == "start":
            root = elem


        # Procesar únicamente al cerrar un safetyreport

        if (
            event == "end"
            and limpiar_tag(elem.tag).lower()
            == "safetyreport"
        ):

            registros_metadata_q1.append(
                {
                    "safetyreportid":
                        obtener_texto(
                            elem,
                            "safetyreportid"
                        ),

                    "safetyreportversion":
                        obtener_texto(
                            elem,
                            "safetyreportversion"
                        ),

                    "receivedate":
                        obtener_texto(
                            elem,
                            "receivedate"
                        ),

                    "receiptdate":
                        obtener_texto(
                            elem,
                            "receiptdate"
                        ),

                    "occurcountry":
                        obtener_texto(
                            elem,
                            "occurcountry"
                        ),

                    "qde_period":
                        "2025Q1",

                    "xml_parte":
                        parte,

                    "xml_archivo":
                        nombre_archivo,
                }
            )

            contador += 1


            # Mostrar avance

            if contador % 50000 == 0:

                print(
                    f"  {contador:,} reportes procesados..."
                )


            # Liberar memoria

            elem.clear()

            if root is not None:
                root.clear()


    conteos_por_archivo.append(
        {
            "archivo": nombre_archivo,
            "parte": parte,
            "n_reportes": contador
        }
    )

    print(
        f"Finalizado: "
        f"{contador:,} reportes"
    )


# 14.4 Construir DataFrame

df_metadata_2025q1 = pd.DataFrame(
    registros_metadata_q1
)

df_conteos_q1 = pd.DataFrame(
    conteos_por_archivo
)


print()
print("=" * 70)
print("RESUMEN DEL PROCESAMIENTO")
print("=" * 70)

display(df_conteos_q1)

print(
    f"Total de reportes extraídos: "
    f"{len(df_metadata_2025q1):,}"
)

ARCHIVOS QUE SERÁN PROCESADOS


,parte,archivo,size_mb
0,1,1_ADR25Q1.xml,640.353873
1,2,2_ADR25Q1.xml,703.806434
2,3,3_ADR25Q1.xml,851.004485



Procesando: 1_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 126,945 reportes

Procesando: 2_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 130,665 reportes

Procesando: 3_ADR25Q1.xml
  50,000 reportes procesados...
  100,000 reportes procesados...
Finalizado: 142,904 reportes

RESUMEN DEL PROCESAMIENTO


,archivo,parte,n_reportes
0,1_ADR25Q1.xml,1,126945
1,2_ADR25Q1.xml,2,130665
2,3_ADR25Q1.xml,3,142904


Total de reportes extraídos: 400,514


El trimestre completo `2025Q1` contiene 400,514 reportes, distribuidos entre los tres fragmentos:

$$
126{,}945+130{,}665+142{,}904=400{,}514.
$$

La lectura incremental funcionó correctamente incluso con más de $2$ GB de XML, así que ya sabemos que esta estrategia es viable para el procesamiento posterior.

## 15. Validación temporal y unicidad de los reportes en el trimestre completo 2025Q1

El procesamiento incremental de los tres archivos XML correspondientes a `2025Q1`
permitió extraer los metadatos de

$$
N=400{,}514
$$

reportes.

Los reportes se distribuyen entre los tres fragmentos como:

$$
126{,}945,
\qquad
130{,}665,
\qquad
142{,}904.
$$

Antes de utilizar estos datos para construir señales trimestrales debemos resolver
dos cuestiones fundamentales:

1. determinar qué variable temporal representa mejor el trimestre de análisis;
2. verificar que los tres archivos XML no contengan reportes duplicados entre sí.



### 15.1 Trimestre del extracto y trimestre de recepción

Cada reporte tiene asociada una procedencia física:

$$
Q_i^{(\mathrm{QDE})}=2025Q1.
$$

Por otro lado, a partir de `receiptdate` podemos construir

$$
Q_i^{(\mathrm{receipt})}.
$$

Si `receiptdate` representa adecuadamente la recepción de la versión incluida en el
extracto, esperaríamos que para la gran mayoría de los reportes se cumpla

$$
Q_i^{(\mathrm{receipt})}
=
Q_i^{(\mathrm{QDE})}.
$$

No se exigirá una igualdad del $100\%$, ya que el sistema FAERS puede incorporar
casos en un extracto posterior debido al procesamiento de versiones y
actualizaciones.

La proporción de concordancia se calculará como

$$
C_{\mathrm{receipt}}
=
\frac{
\#\left\{
i:
Q_i^{(\mathrm{receipt})}
=
Q_i^{(\mathrm{QDE})}
\right\}
}{
N
}.
$$



### 15.2 Papel de `receivedate`

También construiremos el trimestre asociado con `receivedate`:

$$
Q_i^{(\mathrm{received})}.
$$

La exploración preliminar mostró que esta variable puede corresponder a fechas
varios años anteriores al trimestre del extracto.

Esto es compatible con casos que han recibido múltiples actualizaciones o versiones
a lo largo del tiempo.

Por tanto, compararemos:

$$
Q_i^{(\mathrm{received})},
\qquad
Q_i^{(\mathrm{receipt})},
\qquad
Q_i^{(\mathrm{QDE})}.
$$

Esta comparación permitirá decidir posteriormente qué variable debe utilizarse para
el análisis longitudinal de señales.



### 15.3 Diferencia entre fechas

Para cada reporte con ambas fechas completas calcularemos

$$
\Delta_i
=
\texttt{receiptdate}_i
-
\texttt{receivedate}_i.
$$

La cantidad $\Delta_i$ se expresará en días.

Valores grandes de $\Delta_i$ pueden indicar casos cuya información original fue
recibida años antes de la versión presente en el extracto actual.

También analizaremos la relación entre $\Delta_i$ y `safetyreportversion`.



### 15.4 Unicidad de `safetyreportid`

Los tres XML son fragmentos físicos del mismo trimestre.

Por ello debemos comprobar si un mismo

` safetyreportid `

aparece en más de un archivo.

Si definimos

$$
c_i
=
\text{número de veces que aparece el reporte }i,
$$

esperamos idealmente que

$$
c_i=1.
$$

Se calcularán:

- número total de filas;
- número de `safetyreportid` distintos;
- número de identificadores repetidos;
- número de filas adicionales generadas por duplicación;
- posibles reportes presentes en más de un fragmento XML.

Esta verificación es fundamental antes de unir los tres archivos.



### 15.5 Distribución temporal de los fragmentos XML

También analizaremos por separado:

`1_ADR25Q1.xml`,
`2_ADR25Q1.xml`,
`3_ADR25Q1.xml`.

Para cada archivo se calcularán:

- fecha mínima de `receiptdate`;
- fecha máxima de `receiptdate`;
- número de reportes;
- número de fechas distintas.

Esto permitirá determinar si los tres archivos son simplemente divisiones por
tamaño o si existe algún tipo de ordenamiento temporal.

La exploración inicial de los primeros $1{,}000$ reportes sugirió que el contenido
podría encontrarse ordenado por fecha, ya que todos los primeros registros tenían
fechas próximas al inicio del trimestre.


### 15.6 Objetivo de esta sección

Con esta validación queremos responder definitivamente:

1. ¿qué porcentaje de `receiptdate` pertenece realmente a `2025Q1`?;
2. ¿cuáles son los casos que quedan fuera del trimestre?;
3. ¿cuál es la distribución temporal de `receivedate`?;
4. ¿qué diferencias existen entre ambas fechas?;
5. ¿existen `safetyreportid` duplicados entre los tres XML?;
6. ¿los archivos están organizados cronológicamente?;
7. ¿puede utilizarse `receiptdate` como dimensión temporal principal del proyecto?

La elección definitiva de la variable temporal se realizará después de observar
estos resultados.

In [40]:
# 15. Validación temporal y unicidad en 2025Q1 completo


# 15.1 Copia de trabajo

df_q1 = df_metadata_2025q1.copy()


# 15.2 Comprobar valores faltantes y longitud de las fechas

for columna in ["receivedate", "receiptdate"]:

    df_q1[f"{columna}_length"] = (
        df_q1[columna]
        .astype("string")
        .str.len()
    )


print("VALORES FALTANTES")

for columna in ["receivedate", "receiptdate"]:

    n_missing = df_q1[columna].isna().sum()

    print(
        f"{columna}: "
        f"{n_missing:,} "
        f"({100*n_missing/len(df_q1):.4f}%)"
    )


print("\nLONGITUDES DE receivedate")

display(
    df_q1[
        "receivedate_length"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)


print("\nLONGITUDES DE receiptdate")

display(
    df_q1[
        "receiptdate_length"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("n_caracteres")
    .reset_index(name="n_reportes")
)


# 15.3 Convertir fechas completas YYYYMMDD

df_q1["receivedate_dt"] = (
    df_q1["receivedate"]
    .apply(convertir_fecha_faers)
)

df_q1["receiptdate_dt"] = (
    df_q1["receiptdate"]
    .apply(convertir_fecha_faers)
)


print("\nCONVERSIÓN A FECHAS VÁLIDAS")

for columna in [
    "receivedate_dt",
    "receiptdate_dt"
]:

    n_valid = df_q1[columna].notna().sum()

    print(
        f"{columna}: "
        f"{n_valid:,} "
        f"({100*n_valid/len(df_q1):.4f}%)"
    )



# 15.4 Rango temporal global

print("\nRANGO TEMPORAL GLOBAL")

print(
    "receivedate:",
    df_q1["receivedate_dt"].min(),
    "->",
    df_q1["receivedate_dt"].max()
)

print(
    "receiptdate:",
    df_q1["receiptdate_dt"].min(),
    "->",
    df_q1["receiptdate_dt"].max()
)


# 15.5 Crear trimestres calendario

df_q1["receivedate_quarter"] = (
    df_q1["receivedate_dt"]
    .dt.to_period("Q")
    .astype("string")
)

df_q1["receiptdate_quarter"] = (
    df_q1["receiptdate_dt"]
    .dt.to_period("Q")
    .astype("string")
)


# 15.6 Distribución de receiptdate por trimestre

tabla_receipt_q1 = (
    df_q1[
        "receiptdate_quarter"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("trimestre")
    .reset_index(name="n_reportes")
)

tabla_receipt_q1["porcentaje"] = (
    100
    * tabla_receipt_q1["n_reportes"]
    / len(df_q1)
)


print(
    "\nTRIMESTRE CALENDARIO DE receiptdate"
)

display(tabla_receipt_q1)


# 15.7 Concordancia receiptdate vs trimestre QDE

mask_receipt_qde = (
    df_q1["receiptdate_quarter"]
    ==
    df_q1["qde_period"]
)

n_coinciden_qde = mask_receipt_qde.sum()

n_no_coinciden_qde = (
    len(df_q1) - n_coinciden_qde
)


print(
    "\nCONCORDANCIA receiptdate vs qde_period"
)

print(
    f"Coinciden: "
    f"{n_coinciden_qde:,} "
    f"({100*n_coinciden_qde/len(df_q1):.4f}%)"
)

print(
    f"No coinciden: "
    f"{n_no_coinciden_qde:,} "
    f"({100*n_no_coinciden_qde/len(df_q1):.4f}%)"
)


# 15.8 Reportes cuyo receiptdate no pertenece a 2025Q1

receipt_fuera_q1 = (
    df_q1.loc[
        ~mask_receipt_qde,
        [
            "safetyreportid",
            "safetyreportversion",
            "receivedate",
            "receiptdate",
            "receivedate_dt",
            "receiptdate_dt",
            "receivedate_quarter",
            "receiptdate_quarter",
            "xml_parte",
            "xml_archivo"
        ]
    ]
    .sort_values(
        "receiptdate_dt"
    )
    .reset_index(drop=True)
)


print(
    "\nREPORTES CON receiptdate FUERA DE 2025Q1"
)

print(
    f"Número de casos: "
    f"{len(receipt_fuera_q1):,}"
)

display(
    receipt_fuera_q1.head(100)
)


# 15.9 Distribución de receivedate por trimestre

tabla_received_q1 = (
    df_q1[
        "receivedate_quarter"
    ]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("trimestre")
    .reset_index(name="n_reportes")
)

tabla_received_q1["porcentaje"] = (
    100
    * tabla_received_q1["n_reportes"]
    / len(df_q1)
)


print(
    "\nTRIMESTRE CALENDARIO DE receivedate"
)

display(tabla_received_q1)


# 15.10 Diferencia receiptdate - receivedate

mask_ambas = (
    df_q1["receivedate_dt"].notna()
    &
    df_q1["receiptdate_dt"].notna()
)

df_q1.loc[
    mask_ambas,
    "delta_dias"
] = (
    df_q1.loc[
        mask_ambas,
        "receiptdate_dt"
    ]
    -
    df_q1.loc[
        mask_ambas,
        "receivedate_dt"
    ]
).dt.days


print(
    "\nDISTRIBUCIÓN DE receiptdate - receivedate"
)

display(
    df_q1.loc[
        mask_ambas,
        "delta_dias"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 15.11 Coincidencia exacta entre ambas fechas

n_ambas = mask_ambas.sum()

n_fechas_iguales = (
    df_q1.loc[
        mask_ambas,
        "delta_dias"
    ] == 0
).sum()


print(
    "\nCOINCIDENCIA receivedate = receiptdate"
)

print(
    f"Fechas idénticas: "
    f"{n_fechas_iguales:,} de {n_ambas:,} "
    f"({100*n_fechas_iguales/n_ambas:.2f}%)"
)


# 15.12 Unicidad de safetyreportid

n_filas = len(df_q1)

n_ids_unicos = (
    df_q1["safetyreportid"]
    .nunique(dropna=True)
)

n_filas_extra = (
    n_filas - n_ids_unicos
)


print(
    "\nUNICIDAD DE safetyreportid"
)

print(
    f"Filas totales: "
    f"{n_filas:,}"
)

print(
    f"safetyreportid distintos: "
    f"{n_ids_unicos:,}"
)

print(
    f"Filas adicionales por IDs repetidos: "
    f"{n_filas_extra:,}"
)


# 15.13 Identificar IDs repetidos

conteo_ids = (
    df_q1[
        "safetyreportid"
    ]
    .value_counts()
)

ids_repetidos = (
    conteo_ids[
        conteo_ids > 1
    ]
)


print(
    f"IDs que aparecen más de una vez: "
    f"{len(ids_repetidos):,}"
)


# 15.14 Ver si los repetidos aparecen en más de un XML

if len(ids_repetidos) > 0:

    detalle_ids_repetidos = (
        df_q1[
            df_q1[
                "safetyreportid"
            ].isin(
                ids_repetidos.index
            )
        ][
            [
                "safetyreportid",
                "safetyreportversion",
                "receivedate",
                "receiptdate",
                "xml_parte",
                "xml_archivo"
            ]
        ]
        .sort_values(
            [
                "safetyreportid",
                "xml_parte"
            ]
        )
        .reset_index(drop=True)
    )

    print(
        "\nPRIMEROS IDs REPETIDOS"
    )

    display(
        detalle_ids_repetidos.head(100)
    )


    ids_multifragmento = (
        detalle_ids_repetidos
        .groupby(
            "safetyreportid"
        )["xml_parte"]
        .nunique()
    )

    n_multifragmento = (
        ids_multifragmento > 1
    ).sum()


    print(
        f"\nIDs presentes en más de un "
        f"fragmento XML: "
        f"{n_multifragmento:,}"
    )

else:

    detalle_ids_repetidos = pd.DataFrame()

    print(
        "\nNo se detectaron IDs repetidos."
    )


# 15.15 Rango de receiptdate por archivo XML

resumen_temporal_archivo = (
    df_q1
    .groupby(
        [
            "xml_parte",
            "xml_archivo"
        ],
        as_index=False
    )
    .agg(
        n_reportes=(
            "safetyreportid",
            "size"
        ),

        fecha_min=(
            "receiptdate_dt",
            "min"
        ),

        fecha_max=(
            "receiptdate_dt",
            "max"
        ),

        n_fechas_distintas=(
            "receiptdate_dt",
            "nunique"
        )
    )
)


print(
    "\nRANGO DE receiptdate POR ARCHIVO XML"
)

display(resumen_temporal_archivo)


# 15.16 Distribución de safetyreportversion

df_q1["safetyreportversion_num"] = (
    pd.to_numeric(
        df_q1["safetyreportversion"],
        errors="coerce"
    )
)


print(
    "\nDISTRIBUCIÓN DE safetyreportversion"
)

display(
    df_q1[
        "safetyreportversion_num"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 15.17 Casos con las versiones más altas

print(
    "\nREPORTES CON LAS VERSIONES MÁS ALTAS"
)

display(
    df_q1[
        [
            "safetyreportid",
            "safetyreportversion_num",
            "receivedate_dt",
            "receiptdate_dt",
            "delta_dias",
            "xml_archivo"
        ]
    ]
    .sort_values(
        "safetyreportversion_num",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

VALORES FALTANTES
receivedate: 0 (0.0000%)
receiptdate: 0 (0.0000%)

LONGITUDES DE receivedate


,n_caracteres,n_reportes
0,8,400514



LONGITUDES DE receiptdate


,n_caracteres,n_reportes
0,8,400514



CONVERSIÓN A FECHAS VÁLIDAS
receivedate_dt: 400,514 (100.0000%)
receiptdate_dt: 400,514 (100.0000%)

RANGO TEMPORAL GLOBAL
receivedate: 2007-01-12 00:00:00 -> 2025-03-31 00:00:00
receiptdate: 2024-09-27 00:00:00 -> 2025-03-31 00:00:00

TRIMESTRE CALENDARIO DE receiptdate


...

,xml_parte,xml_archivo,n_reportes,fecha_min,fecha_max,n_fechas_distintas
0,1,1_ADR25Q1.xml,126945,2024-09-27,2025-01-31,34
1,2,2_ADR25Q1.xml,130665,2025-02-01,2025-02-28,28
2,3,3_ADR25Q1.xml,142904,2025-03-01,2025-03-31,31



DISTRIBUCIÓN DE safetyreportversion


,valor
count,400514.000000
mean,1.865380
std,2.714392
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
90%,3.000000
95%,5.000000
99%,13.000000



REPORTES CON LAS VERSIONES MÁS ALTAS


,safetyreportid,safetyreportversion_num,receivedate_dt,receiptdate_dt,delta_dias,xml_archivo
0,19454468,146,2021-06-23,2025-03-31,1377.0,3_ADR25Q1.xml
1,16630995,100,2019-07-25,2025-01-17,2003.0,1_ADR25Q1.xml
2,19035584,93,2021-03-20,2025-02-26,1439.0,2_ADR25Q1.xml
3,18703655,93,2021-01-05,2025-02-13,1500.0,2_ADR25Q1.xml
4,18665055,90,2020-12-25,2025-03-27,1553.0,3_ADR25Q1.xml
5,19012827,89,2021-03-16,2025-03-25,1470.0,3_ADR25Q1.xml
6,18566067,88,2020-12-01,2025-03-28,1578.0,3_ADR25Q1.xml
7,20121217,87,2021-11-27,2025-03-20,1209.0,3_ADR25Q1.xml
8,22298350,87,2023-05-09,2025-03-21,682.0,3_ADR25Q1.xml
9,18853661,85,2021-02-05,2025-02-27,1483.0,2_ADR25Q1.xml


Estos resultados ya nos permiten tomar una decisión temporal sustentada para el proyecto.

La evidencia es muy clara: `receiptdate` coincide con el trimestre del extracto en **400,511 de 400,514 reportes**, es decir, $99.9993\%$.


En cambio, `receivedate` se extiende desde 2007 hasta 2025 y solamente el 79.33% pertenece a `2025Q1`. Esto es coherente con la estructura Case/Version de FAERS: la FDA indica que `receivedate` permite conocer cuándo se recibió inicialmente el caso, mientras que `receiptdate` corresponde a la fecha de recepción de la versión del caso; además, el QDE proporciona la versión más actual disponible. 

También descubrimos algo muy valioso: los tres archivos XML **no son fragmentaciones arbitrarias por tamaño**, sino que prácticamente corresponden a los tres meses del trimestre:

$$
\begin{aligned}
1_\text{ADR25Q1} &\rightarrow \text{enero},\\
2_\text{ADR25Q1} &\rightarrow \text{febrero},\\
3_\text{ADR25Q1} &\rightarrow \text{marzo}.
\end{aligned}
$$

Solo hay tres casos excepcionales anteriores a 2025Q1 en el primer archivo. Además, los **400,514 `safetyreportid` son únicos**, por lo que no hay solapamiento entre los tres fragmentos de Q1.

## 16. Definición de la dimensión temporal del análisis

La validación realizada sobre los $400{,}514$ reportes del trimestre completo
`2025Q1` permite establecer formalmente la variable temporal que se utilizará en
el proyecto.

Se compararon tres referencias:

$$
Q_i^{(\mathrm{received})},
\qquad
Q_i^{(\mathrm{receipt})},
\qquad
Q_i^{(\mathrm{QDE})}.
$$

Los resultados mostraron comportamientos claramente diferentes.


### 16.1 `receivedate`: fecha histórica del caso

La variable `receivedate` presentó fechas desde

$$
2007\text{-}01\text{-}12
$$

hasta

$$
2025\text{-}03\text{-}31.
$$

Solamente aproximadamente el

$$
79.33\%
$$

de los reportes presentó un `receivedate` correspondiente a `2025Q1`.

Además, algunos casos con versiones elevadas presentaron diferencias de varios años
entre `receivedate` y `receiptdate`.

Por ejemplo, se encontraron reportes con más de cien versiones y fechas iniciales
varios años anteriores al trimestre del extracto.

Esto indica que `receivedate` contiene información importante sobre la historia
temporal del caso, pero no representa adecuadamente el trimestre en el que la versión
analizada fue recibida.

Por tanto, `receivedate` se conservará como una variable auxiliar que permitirá
caracterizar la antigüedad del caso y su historial de actualizaciones.



### 16.2 `receiptdate`: fecha temporal principal

La variable `receiptdate` presentó una correspondencia prácticamente exacta con el
trimestre del extracto.

De los

$$
400{,}514
$$

reportes analizados,

$$
400{,}511
$$

presentaron

$$
Q_i^{(\mathrm{receipt})}
=
2025Q1.
$$

Esto representa una concordancia de aproximadamente

$$
99.9993\%.
$$

Solamente tres reportes presentaron `receiptdate` anterior al trimestre del extracto.

Por tanto, definiremos la fecha analítica principal como

$$
T_i
=
\texttt{receiptdate}_i.
$$

El trimestre analítico será

$$
Q_i
=
\operatorname{Quarter}(T_i).
$$

En el código esta variable se denominará

`analysis_quarter`.



### 16.3 Papel de `qde_period`

El trimestre del archivo de procedencia se conservará mediante

`qde_period`.

Esta variable no sustituirá a `receiptdate`, sino que se utilizará como mecanismo de
trazabilidad y control de calidad.

Definiremos

$$
M_i
=
\mathbb{I}
\left(
Q_i^{(\mathrm{receipt})}
=
Q_i^{(\mathrm{QDE})}
\right),
$$

donde

$$
M_i=1
$$

indica concordancia entre ambas referencias temporales.

Los casos con

$$
M_i=0
$$

no serán eliminados de la base original. Se conservarán y serán identificados mediante
una bandera de control de calidad.



### 16.4 Organización mensual de los archivos XML

La inspección del trimestre completo mostró además que los tres archivos XML están
organizados cronológicamente:

- `1_ADR25Q1.xml`: principalmente enero de 2025;
- `2_ADR25Q1.xml`: febrero de 2025;
- `3_ADR25Q1.xml`: marzo de 2025.

Esta organización explica por qué los primeros $1{,}000$ reportes analizados
previamente correspondían casi exclusivamente al inicio del trimestre.

Por tanto, los primeros registros de un XML no deben considerarse una muestra
aleatoria del trimestre.



### 16.5 Unicidad dentro del trimestre

Se verificó también que

$$
N_{\mathrm{filas}}
=
N_{\mathrm{safetyreportid}}
=
400{,}514.
$$

Por lo tanto, no existen `safetyreportid` repetidos entre los tres fragmentos XML de
`2025Q1`.

Esta conclusión se refiere únicamente a la unicidad **dentro del trimestre**.

Todavía será necesario comprobar si un mismo `safetyreportid` puede aparecer en
diferentes trimestres debido a nuevas versiones del mismo caso.

Esta será una validación particularmente importante antes de construir los conteos
longitudinales de señales.



### 16.6 Regla temporal del proyecto

A partir de este punto se adoptará la siguiente convención:

$$
\boxed{
\texttt{analysis\_date}
=
\texttt{receiptdate}
}
$$

y

$$
\boxed{
\texttt{analysis\_quarter}
=
\operatorname{Quarter}
(\texttt{receiptdate})
}
$$

mientras que:

- `qde_period` indicará el extracto de procedencia;
- `receivedate` conservará la fecha histórica inicial;
- `delta_dias` medirá la diferencia entre ambas fechas;
- `temporal_match` indicará si `analysis_quarter` coincide con `qde_period`.

Esta separación permitirá mantener simultáneamente la dimensión temporal del análisis,
la historia del caso y la trazabilidad del archivo fuente.

In [42]:
# 16. Definición formal de la dimensión temporal


# 16.1 Fecha y trimestre analítico

df_q1["analysis_date"] = (
    df_q1["receiptdate_dt"]
)

df_q1["analysis_quarter"] = (
    df_q1["analysis_date"]
    .dt.to_period("Q")
    .astype("string")
)


# 16.2 Concordancia con el trimestre QDE

df_q1["temporal_match"] = (
    df_q1["analysis_quarter"]
    ==
    df_q1["qde_period"]
)


# 16.3 Antigüedad del caso

df_q1["case_history_days"] = (
    df_q1["analysis_date"]
    -
    df_q1["receivedate_dt"]
).dt.days


# 16.4 Indicador de caso actualizado
# Si receiptdate > receivedate, la versión presente en el extracto fue recibida después de la fecha inicial.

df_q1["has_followup_history"] = (
    df_q1["case_history_days"] > 0
)


# 16.5 Ventana temporal del estudio

trimestres_estudio = [
    "2025Q1",
    "2025Q2",
    "2025Q3",
    "2025Q4",
    "2026Q1",
    "2026Q2",
]


df_q1["in_study_window"] = (
    df_q1["analysis_quarter"]
    .isin(trimestres_estudio)
)


# 16.6 Resumen de validación

print("DEFINICIÓN TEMPORAL FINAL PARA 2025Q1")

print(
    f"Reportes totales: "
    f"{len(df_q1):,}"
)

print(
    f"analysis_quarter = qde_period: "
    f"{df_q1['temporal_match'].sum():,} "
    f"({100*df_q1['temporal_match'].mean():.4f}%)"
)

print(
    f"analysis_quarter != qde_period: "
    f"{(~df_q1['temporal_match']).sum():,} "
    f"({100*(~df_q1['temporal_match']).mean():.4f}%)"
)

print(
    f"Reportes dentro de la ventana "
    f"2025Q1-2026Q2: "
    f"{df_q1['in_study_window'].sum():,}"
)

print(
    f"Reportes fuera de la ventana: "
    f"{(~df_q1['in_study_window']).sum():,}"
)


# 16.7 Historia previa del caso

print(
    "\nHISTORIA TEMPORAL DEL CASO"
)

n_followup = (
    df_q1["has_followup_history"]
    .sum()
)

print(
    f"Reportes con receiptdate > receivedate: "
    f"{n_followup:,} "
    f"({100*n_followup/len(df_q1):.2f}%)"
)

print(
    f"Reportes con ambas fechas iguales: "
    f"{(~df_q1['has_followup_history']).sum():,} "
    f"({100*(~df_q1['has_followup_history']).mean():.2f}%)"
)


# 16.8 Distribución de case_history_days

print(
    "\nDISTRIBUCIÓN DE case_history_days"
)

display(
    df_q1[
        "case_history_days"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(name="valor")
)


# 16.9 Casos con discordancia temporal

df_temporal_mismatch_q1 = (
    df_q1[
        ~df_q1["temporal_match"]
    ][
        [
            "safetyreportid",
            "safetyreportversion",
            "receivedate_dt",
            "analysis_date",
            "receivedate_quarter",
            "analysis_quarter",
            "qde_period",
            "case_history_days",
            "xml_archivo"
        ]
    ]
    .sort_values("analysis_date")
    .reset_index(drop=True)
)


print(
    "\nCASOS CON DISCORDANCIA TEMPORAL"
)

display(df_temporal_mismatch_q1)


# 16.10 Verificaciones automáticas

print(
    "\nVERIFICACIONES"
)


# analysis_date debe coincidir con receiptdate_dt
check_fecha = (
    df_q1["analysis_date"]
    .equals(
        df_q1["receiptdate_dt"]
    )
)


# No debe haber analysis_date faltante en Q1
check_missing = (
    df_q1["analysis_date"]
    .isna()
    .sum() == 0
)


# safetyreportid debe seguir siendo único
check_unique = (
    df_q1["safetyreportid"]
    .is_unique
)


print(
    "analysis_date coincide con receiptdate_dt:",
    check_fecha
)

print(
    "analysis_date sin valores faltantes:",
    check_missing
)

print(
    "safetyreportid único dentro de 2025Q1:",
    check_unique
)

DEFINICIÓN TEMPORAL FINAL PARA 2025Q1
Reportes totales: 400,514
analysis_quarter = qde_period: 400,511 (99.9993%)
analysis_quarter != qde_period: 3 (0.0007%)
Reportes dentro de la ventana 2025Q1-2026Q2: 400,511
Reportes fuera de la ventana: 3

HISTORIA TEMPORAL DEL CASO
Reportes con receiptdate > receivedate: 125,008 (31.21%)
Reportes con ambas fechas iguales: 275,506 (68.79%)

DISTRIBUCIÓN DE case_history_days


,valor
count,400514.000000
mean,91.837646
std,310.149612
min,0.000000
25%,0.000000
50%,0.000000
75%,20.000000
90%,215.000000
95%,551.000000
99%,1646.000000



CASOS CON DISCORDANCIA TEMPORAL


,safetyreportid,safetyreportversion,receivedate_dt,analysis_date,receivedate_quarter,analysis_quarter,qde_period,case_history_days,xml_archivo
0,24918460,1,2024-09-27,2024-09-27,2024Q3,2024Q3,2025Q1,0,1_ADR25Q1.xml
1,24717255,1,2024-12-10,2024-12-10,2024Q4,2024Q4,2025Q1,0,1_ADR25Q1.xml
2,24789549,1,2024-12-30,2024-12-30,2024Q4,2024Q4,2025Q1,0,1_ADR25Q1.xml



VERIFICACIONES
analysis_date coincide con receiptdate_dt: True
analysis_date sin valores faltantes: True
safetyreportid único dentro de 2025Q1: True


# 17. Resumen y conclusiones

Este notebook permitió caracterizar la estructura de los archivos FAERS y establecer
las principales decisiones metodológicas que se utilizarán en las siguientes etapas
del proyecto.

Los datos disponibles comprenden seis trimestres,

$$
2025Q1,\ldots,2026Q2,
$$

con un total de $18$ archivos XML y aproximadamente $12.75$ GB de información. Debido
a su tamaño, se determinó que el procesamiento deberá realizarse mediante lectura
incremental con `iterparse`.

La exploración de los reportes mostró que un mismo `safetyreportid` puede contener
múltiples medicamentos y múltiples reacciones. En una muestra inicial de $1000$
reportes se encontraron $6104$ registros `<drug>` y $3427$ registros `<reaction>`.

Para el análisis principal se utilizarán los medicamentos clasificados como
`Suspect`. El identificador analítico del medicamento será

$$
\texttt{drug\_key}
=
\begin{cases}
\texttt{activesubstancename}, & \text{si está disponible},\\
\texttt{medicinalproduct}, & \text{en otro caso}.
\end{cases}
$$

Se comprobó además que un mismo medicamento puede aparecer varias veces dentro de un
reporte debido a diferencias en dosis, indicación u otras características. Por ello,
la unidad analítica de exposición se definirá como

$$
(\texttt{safetyreportid},\texttt{drug\_key}),
$$

de manera que cada medicamento contribuya una sola vez por reporte.

El análisis temporal del trimestre completo `2025Q1`, formado por $400{,}514$
reportes, mostró que `receiptdate` coincide con el trimestre del extracto en el

$$
99.9993\%
$$

de los casos. En consecuencia, se definió

$$
\boxed{\texttt{analysis\_date}=\texttt{receiptdate}}
$$

y

$$
\boxed{
\texttt{analysis\_quarter}
=
\operatorname{Quarter}(\texttt{receiptdate})
}.
$$

`receivedate` se conservará como referencia histórica del caso y `qde_period` como
variable de procedencia y control de calidad.

Finalmente, se verificó que los $400{,}514$ reportes de `2025Q1` tienen
`safetyreportid` únicos dentro del trimestre.

Por tanto, las principales decisiones que se trasladan a los siguientes notebooks
son:

- procesamiento incremental de los XML;
- selección de medicamentos `Suspect`;
- uso preferente de `activesubstancename` para construir `drug_key`;
- deduplicación a nivel reporte--medicamento y, posteriormente,
  reporte--medicamento--evento;
- uso de `receiptdate` como referencia temporal principal;
- conservación de `receivedate` y `qde_period` para trazabilidad y control.

El siguiente notebook extenderá el análisis de metadatos a los seis trimestres para
determinar si un mismo `safetyreportid` reaparece en distintos periodos con nuevas
`safetyreportversion`, antes de realizar la extracción masiva de medicamentos y
reacciones.

# Cuestionario

El objetivo de este cuestionario es verificar la comprensión de los conceptos,
variables y decisiones metodológicas desarrolladas en este notebook.

Responde cada pregunta **con tus propias palabras**. Cuando sea necesario, utiliza
ecuaciones, ejemplos o fragmentos de código para justificar tu respuesta.


### 1. ¿Qué es FAERS y cuál es su utilidad en farmacovigilancia?

Explica brevemente:

- qué significa FAERS;
- qué tipo de información contiene;
- qué es un evento adverso;
- por qué una asociación encontrada en FAERS no implica necesariamente que un
  medicamento haya causado el evento.



### 2. ¿Cómo está organizado un reporte dentro de los archivos XML de FAERS?

Explica la relación entre los elementos:

`<safetyreport>`, `<patient>`, `<drug>` y `<reaction>`.

En particular, explica por qué un mismo `safetyreportid` puede estar asociado con
varios medicamentos y varias reacciones.

Puedes representar esquemáticamente la estructura como

$$
\text{safetyreport}
\longrightarrow
\text{patient}
\longrightarrow
\begin{cases}
\text{drug}_1,\ldots,\text{drug}_{m_i},\\
\text{reaction}_1,\ldots,\text{reaction}_{n_i}.
\end{cases}
$$



### 3. Explica el significado de las siguientes variables

Describe con tus propias palabras qué información contiene cada una:

- `safetyreportid`;
- `safetyreportversion`;
- `occurcountry`;
- `medicinalproduct`;
- `activesubstancename`;
- `drugcharacterization`;
- `reactionmeddrapt`;
- `receivedate`;
- `receiptdate`.

Indica además cuáles serán especialmente importantes para el análisis temporal,
geográfico y de pares medicamento--evento.



### 4. ¿Cuál es la diferencia entre `medicinalproduct` y `activesubstancename`?

Explica por qué dos variables diferentes son necesarias.

Utiliza alguno de los ejemplos observados en el notebook, como

`DUPIXENT` y `DUPILUMAB`

o

`REVLIMID` y `LENALIDOMIDE`.

Finalmente, explica por qué se decidió construir la variable `drug_key` dando
prioridad a `activesubstancename`.



### 5. ¿Qué papel tiene `drugcharacterization` en el proyecto?

Explica qué significa que un medicamento esté clasificado como `Suspect` y por qué
no sería conveniente tratar automáticamente todos los medicamentos de un reporte
como si tuvieran el mismo papel respecto al evento adverso.

¿Qué valor de `drugcharacterization` se utiliza para seleccionar la cohorte
principal de medicamentos en este notebook?



### 6. ¿Por qué es necesario deduplicar los medicamentos dentro de un reporte?

En la muestra estudiada se encontraron

$$
2938
$$

registros `<drug>` clasificados como `Suspect`, pero solamente

$$
1589
$$

pares únicos reporte--medicamento.

Explica por qué estas dos cantidades son diferentes.

¿Qué problema produciría utilizar directamente las $2938$ filas como exposiciones
independientes?

Explica también qué enseñó el caso en el que `TOCILIZUMAB` apareció $85$ veces
dentro de un mismo reporte.



### 7. ¿Cuál es la unidad analítica definida para los medicamentos?

Explica qué representa la combinación

$$
(\texttt{safetyreportid},\texttt{drug\_key})
$$

y por qué cada combinación debe aparecer una sola vez en la tabla analítica.

Posteriormente, cuando se incorporen las reacciones adversas, ¿cuál será la nueva
unidad analítica?

Escribe la combinación de variables correspondiente.



### 8. ¿Cuál es la diferencia entre `receivedate`, `receiptdate` y `qde_period`?

Explica qué información temporal aporta cada variable.

En `2025Q1` se analizaron

$$
400{,}514
$$

reportes y en

$$
400{,}511
$$

casos el trimestre obtenido a partir de `receiptdate` coincidió con `qde_period`.

Calcula el porcentaje de concordancia e interpreta este resultado.

¿Por qué se decidió utilizar

$$
\texttt{analysis\_date}
=
\texttt{receiptdate}
$$

en lugar de `receivedate`?



### 9. ¿Qué información proporciona `safetyreportversion`?

Explica qué significa encontrar, por ejemplo, un reporte con:

- `safetyreportversion = 1`;
- `safetyreportversion = 57`;
- `safetyreportversion = 146`.

¿Por qué será importante investigar si el mismo `safetyreportid` aparece nuevamente
en diferentes trimestres antes de construir las señales longitudinales?



### 10. Describe el pipeline que se ha construido hasta este momento

Sin consultar el código, describe con tus propias palabras las etapas necesarias
para pasar desde los archivos XML originales hasta una futura tabla de pares
medicamento--evento.

Tu explicación debe incluir al menos los siguientes conceptos:

- archivos QDE;
- lectura incremental;
- `safetyreportid`;
- medicamentos `Suspect`;
- `drug_key`;
- términos MedDRA;
- deduplicación;
- `receiptdate`;
- trimestre analítico;
- pares medicamento--evento.

Puedes ayudarte con el siguiente esquema y completarlo:

$$
\text{XML}
\rightarrow
\text{safetyreport}
\rightarrow
\boxed{?}
\rightarrow
\boxed{?}
\rightarrow
\text{limpieza}
\rightarrow
\text{deduplicación}
\rightarrow
\boxed{?}
$$



## Indicaciones

Las respuestas deben centrarse en **explicar el porqué de cada decisión**, no
solamente en definir los términos.

En particular, se evaluará que puedas distinguir correctamente:

1. una fila XML de una unidad analítica;
2. un producto comercial de una sustancia activa;
3. un medicamento sospechoso de uno concomitante;
4. `receivedate` de `receiptdate`;
5. un caso de una versión del caso;
6. una asociación estadística de una relación causal.